# Important!

<span style="color: red;">This file code may contain some banned packages. None of the models were used.</span>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from functools import partial
import importlib

import src.run_hog
import src.features
import src.data
import src.kernels
import src.klr
import src.metrics
import src.preprocessing
import src.utils
from src.data import encode_labels, load_data, train_val_split
from src.kernels import (
    estimate_gamma,
    estimate_gamma_l1,
    gaussian_kernel_matrix,
    laplacian_kernel_matrix,
    linear_kernel_matrix,
    chi2_kernel_matrix,
 )
from src.klr import KernelLogisticRegression
from src.metrics import accuracy
from src.preprocessing import StandardiseData, PCA
from src.utils import show_vector_image
from src.features import compute_hog_features
from src.run_hog import main 



In [ ]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

scaler = StandardiseData().fit(X_tr.astype(np.float32))
X_tr_s = scaler.transform(X_tr.astype(np.float32))
X_te_s = scaler.transform(X_te.astype(np.float32))

pca = PCA(n_components=512, whiten=False).fit(X_tr.astype(np.float32))
X_tr_p = pca.transform(X_tr_s.astype(np.float32))
X_te_p = pca.transform(X_te_s.astype(np.float32))

n_classes = len(np.unique(y_tr))

X_train, X_val, y_train, y_val = train_val_split(X_tr_p, y_tr, test_size=0.3, seed=20)

## Linear kernel
Quick grid search over learning rate and regularisation.

In [ ]:
lrs = [0.3, 0.2, 0.1]
lams = [1e-5, 1e-4, 1e-3]

best_lr, best_lam = lrs[0], lams[0]
best_acc = 0.0

for lr, lam in product(lrs, lams):
    linear_model = KernelLogisticRegression(
        n_classes=n_classes,
        kernel_fn=linear_kernel_matrix,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = linear_model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam = lr, lam

print("best:", {"lr": best_lr, "lambda": best_lam, "val_acc": best_acc})

## Train and write submission (linear)

In [ ]:
best_linear_model = KernelLogisticRegression(
    n_classes=n_classes,
    kernel_fn=linear_kernel_matrix,
    lr=best_lr,
    epochs=500,
    lam=best_lam,
).fit(X_tr_p, y_tr)

yte_int, _ = best_linear_model.predict(X_te_p)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")

## Gaussian (RBF) kernel
Grid search over learning rate, regularisation, and gamma (RBF width).

In [ ]:
X_train, X_val, y_train, y_val = train_val_split(X_tr_s, y_tr, test_size=0.2, seed=20)

lrs = [5e-1, 1e-1]
lams = [5e-4, 1e-4, 5e-4]

gamma0 = estimate_gamma(X_train)
gammas = [gamma0 / 5, gamma0, gamma0 * 5]

best_lr, best_lam, best_gamma = lrs[0], lams[0], gammas[0]
best_acc = 0.0

for lr, lam, gamma in product(lrs, lams, gammas):
    kernel_fn = partial(gaussian_kernel_matrix, gamma=gamma)
    model = KernelLogisticRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, gamma={gamma:.3e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam, best_gamma = lr, lam, gamma

print("best:", {"lr": best_lr, "lambda": best_lam, "gamma": best_gamma, "val_acc": best_acc})

In [ ]:
best_gaussian_model = KernelLogisticRegression(
    n_classes=n_classes,
    kernel_fn=partial(gaussian_kernel_matrix, gamma=best_gamma),
    lr=best_lr,
    epochs=500,
    lam=best_lam,
).fit(X_tr_p, y_tr)

yte_int, _ = best_gaussian_model.predict(X_te_p)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/resubmission.csv", index_label="Id")

In [ ]:
importlib.reload(src.klr)
importlib.reload(src.preprocessing)
from src.klr import KernelLogisticRegression
from src.preprocessing import StandardiseData, PCA

In [ ]:
X_tr_p[5].max()

In [ ]:
# how to copy 
idx = 106

np.clip(X_tr[idx])

In [ ]:
idx = 106
show_vector_image(X_tr[idx], index=idx, title="reshape (3,32,32) then transpose")
# print(y_val[idx])
# lc, lps = best_linear_model.predict(X_tr_p[idx : idx + 1])
# print(lc[0])
# print(lps[0])
# gc, gps = best_gaussian_model.predict(X_val[idx : idx + 1])
# print(gc[0])
# print(gps[0])

HOG with SVM

In [ ]:
# HOG + SVM (organized): delegate to src/run_svm_hog.py
import importlib
import src.run_svm_hog

importlib.reload(src.run_svm_hog)

from src.run_svm_hog import HogParams, SvmParams, prepare_hog_features, fit_linear_svm_gridsearch

# 1) Prepare HOG features
X_hog_s, X_test_hog_s, y_int, inv_map = prepare_hog_features(
    "data/",
    hog_params=HogParams(
        orientations=12,
        pixels_per_cell=(6, 6),
        cells_per_block=(3, 3),
    ),
)
print(f"HOG train shape: {X_hog_s.shape}, test shape: {X_test_hog_s.shape}")

# 2) Fit linear SVM with a tiny grid-search
final_svm, best_C, best_acc = fit_linear_svm_gridsearch(
    X_hog_s,
    y_int,
    svm_params=SvmParams(
        Cs=(0.03, 0.1, 0.3, 1.0, 3.0, 10.0),
        test_size=0.2,
        seed=42,
        max_iter_search=5000,
        max_iter_final=8000,
    ),
)
print(f"Best linear SVM: C={best_C} | val_acc={best_acc:.4f}")

In [ ]:
# Generate submission with HOG + linear SVM
import importlib
import src.run_svm_hog

importlib.reload(src.run_svm_hog)

from src.run_svm_hog import write_submission

yte_int = final_svm.predict(X_test_hog_s)
out = write_submission(
    yte_int=yte_int,
    inv_map=inv_map,
    out_path="results/submission_hog_svm.csv",
)
print(f"Saved to {out}")
print(f"Used linear SVM hyperparameter: C={best_C}")

HOG with KLR 

In [ ]:
# HOG + KLR (Chi^2 kernel) - quick version
import numpy as np
import pandas as pd

from itertools import product
from functools import partial
import importlib

import src.features
import src.data
import src.kernels
import src.klr
import src.metrics
import src.preprocessing

importlib.reload(src.features)
importlib.reload(src.data)
importlib.reload(src.kernels)
importlib.reload(src.klr)
importlib.reload(src.metrics)
importlib.reload(src.preprocessing)

from src.data import load_data, train_val_split, encode_labels
from src.features import compute_hog_features
from src.kernels import chi2_kernel_matrix
from src.klr import KernelLogisticRegression
from src.metrics import accuracy
from src.preprocessing import StandardiseData

DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

print("Extracting HOG features...")
X_hog = compute_hog_features(
    X_raw,
    orientations=12,
    pixels_per_cell=(6, 6),
    cells_per_block=(3, 3),
)
X_test_hog = compute_hog_features(
    X_test_raw,
    orientations=12,
    pixels_per_cell=(6, 6),
    cells_per_block=(3, 3),
)
print(f"HOG train shape: {X_hog.shape}, test shape: {X_test_hog.shape}")

# Standardize HOG features (helps optimization)
scaler = StandardiseData().fit(X_hog.astype(np.float32))
X_hog_s = scaler.transform(X_hog.astype(np.float32))
X_test_hog_s = scaler.transform(X_test_hog.astype(np.float32))

# Train on a smaller subset for speed
subset_size = 600
X_sub = X_hog_s[:subset_size]
y_sub = y_int[:subset_size]
n_classes = len(np.unique(y_int))

X_train, X_val, y_train, y_val = train_val_split(X_sub, y_sub, test_size=0.2, seed=42)

# Small grid search (keep it small; each run builds big kernel matrices)
gammas = [0.1, 0.5, 1.0]
lrs = [0.1, 0.05]
lams = [1e-3, 5e-4]

best_gamma, best_lr, best_lam = gammas[0], lrs[0], lams[0]
best_acc = -1.0

for gamma, lr, lam in product(gammas, lrs, lams):
    kernel_fn = partial(chi2_kernel_matrix, gamma=gamma)
    model = KernelLogisticRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=60,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = model.predict(X_val)
    acc = accuracy(y_val, pred_val)
    print(f"gamma={gamma:.2g} lr={lr:.2g} lam={lam:.0e} val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_gamma, best_lr, best_lam = gamma, lr, lam

print(f"\nBest params: gamma={best_gamma} lr={best_lr} lam={best_lam} val_acc={best_acc:.4f}")

# Fit final model on the subset with best params
final_klr = KernelLogisticRegression(
    n_classes=n_classes,
    kernel_fn=partial(chi2_kernel_matrix, gamma=best_gamma),
    lr=best_lr,
    epochs=100,
    lam=best_lam,
    patience=8,
).fit(X_sub, y_sub)

# Predict on the full test set (kernel between test and subset)
yte_int, _ = final_klr.predict(X_test_hog_s)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_hog_klr.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## HOG + KRR (Kernel Ridge Regression)
Kernel Ridge Regression (one-vs-rest) using a Laplacian kernel on HOG features.
This is a *quick* version: small subset + tiny grid over $(\lambda, \gamma)$.

In [ ]:
# HOG + KRR (≈5 min budget) using Laplacian kernel
import numpy as np
import pandas as pd
from itertools import product

# Try to reuse HOG features if they already exist (from the HOG+KLR cell).
if "X_hog_s" in globals() and "X_test_hog_s" in globals() and "y_int" in globals() and "inv_map" in globals():
    X_hog_s_krr, X_test_hog_s_krr, y_int_krr, inv_map_krr = X_hog_s, X_test_hog_s, y_int, inv_map
else:
    # Fallback: compute HOG + standardize
    from src.run_svm_hog import HogParams, prepare_hog_features
    X_hog_s_krr, X_test_hog_s_krr, y_int_krr, inv_map_krr = prepare_hog_features(
        "data/",
        hog_params=HogParams(orientations=12, pixels_per_cell=(6, 6), cells_per_block=(3, 3)),
    )

n_classes = int(len(np.unique(y_int_krr)))

# Subset for speed (kernel methods are O(n^2))
subset_size = 1200
X_sub = X_hog_s_krr[:subset_size].astype(np.float32, copy=False)
y_sub = y_int_krr[:subset_size]

X_train, X_val, y_train, y_val = train_val_split(X_sub, y_sub, test_size=0.2, seed=42)

def one_hot(y: np.ndarray, C: int) -> np.ndarray:
    Y = np.zeros((y.shape[0], C), dtype=np.float32)
    Y[np.arange(y.shape[0]), y.astype(int)] = 1.0
    return Y

def l1_dist_matrix(X: np.ndarray, Z: np.ndarray, *, chunk_size: int = 128) -> np.ndarray:
    """Compute pairwise L1 distances with chunking: returns (n, m)."""
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n = X.shape[0]
    m = Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_size):
        i1 = min(i0 + chunk_size, n)
        Xi = X[i0:i1]
        D[i0:i1] = np.sum(np.abs(Xi[:, None, :] - Z[None, :, :]), axis=2, dtype=np.float32)
    return D

# Grid over ridge lambda and Laplacian gamma
gamma0 = estimate_gamma_l1(X_train, sample=min(350, X_train.shape[0]), seed=0)
gammas = [gamma0 / 3, gamma0, gamma0 * 3]
lams = [1e-3, 3e-3, 1e-2]  # ridge regularization

# Precompute distances once (saves time during gamma grid)
print("Precomputing L1 distance matrices...")
D_tr = l1_dist_matrix(X_train, X_train, chunk_size=96)
D_val = l1_dist_matrix(X_val, X_train, chunk_size=96)

Y_tr = one_hot(y_train, n_classes)
I = np.eye(D_tr.shape[0], dtype=np.float32)

best = {"acc": -1.0}
for gamma in gammas:
    K_tr = np.exp(-float(gamma) * D_tr).astype(np.float32, copy=False)
    K_val = np.exp(-float(gamma) * D_val).astype(np.float32, copy=False)
    for lam in lams:
        A = np.linalg.solve(K_tr + float(lam) * I, Y_tr)
        pred = (K_val @ A).argmax(axis=1)
        acc = float(np.mean(pred == y_val))
        print(f"lam={lam:.1e} gamma={gamma:.3e} val_acc={acc:.4f}")
        if acc > best["acc"]:
            best = {"acc": acc, "lam": float(lam), "gamma": float(gamma)}

print("Best KRR params:", best)

# Train final KRR on subset and predict test
print("Training final KRR on subset...")
D_sub = l1_dist_matrix(X_sub, X_sub, chunk_size=96)
K_sub = np.exp(-best["gamma"] * D_sub).astype(np.float32, copy=False)
Y_sub = one_hot(y_sub, n_classes)
A_sub = np.linalg.solve(K_sub + best["lam"] * np.eye(K_sub.shape[0], dtype=np.float32), Y_sub)

print("Computing test cross-distances...")
D_test = l1_dist_matrix(X_test_hog_s_krr.astype(np.float32, copy=False), X_sub, chunk_size=96)
K_test = np.exp(-best["gamma"] * D_test).astype(np.float32, copy=False)
scores_test = K_test @ A_sub
yte_int = scores_test.argmax(axis=1)
yte = np.array([inv_map_krr[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_hog_krr.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## HOG KRR LAPLACIAN
Laplacian kernel is an RBF-like kernel but using **L1 distance** instead of L2:
$$k(x,z)=\exp(-\gamma\|x-z\|_1).$$
- It’s positive definite and often works well when features are sparse / histogram-like.
- A common alternative (often very strong on HOG) is the **Gaussian (RBF) kernel**: $\exp(-\gamma\|x-z\|_2^2)$.

In [ ]:
# HOG + KRR with Gaussian (RBF) kernel (often a strong baseline)
import numpy as np
import pandas as pd
from itertools import product

# Reuse HOG features if available
if "X_hog_s" in globals() and "X_test_hog_s" in globals() and "y_int" in globals() and "inv_map" in globals():
    X_hog_s_rbf, X_test_hog_s_rbf, y_int_rbf, inv_map_rbf = X_hog_s, X_test_hog_s, y_int, inv_map
else:
    from src.run_svm_hog import HogParams, prepare_hog_features
    X_hog_s_rbf, X_test_hog_s_rbf, y_int_rbf, inv_map_rbf = prepare_hog_features(
        "data/",
        hog_params=HogParams(orientations=12, pixels_per_cell=(6, 6), cells_per_block=(3, 3)),
    )

n_classes = int(len(np.unique(y_int_rbf)))

subset_size = 1400  # ~5 min budget depending on machine
X_sub = X_hog_s_rbf[:subset_size].astype(np.float32, copy=False)
y_sub = y_int_rbf[:subset_size]

X_train, X_val, y_train, y_val = train_val_split(X_sub, y_sub, test_size=0.2, seed=42)

def one_hot(y: np.ndarray, C: int) -> np.ndarray:
    Y = np.zeros((y.shape[0], C), dtype=np.float32)
    Y[np.arange(y.shape[0]), y.astype(int)] = 1.0
    return Y

def sq_dists(X: np.ndarray, Z: np.ndarray) -> np.ndarray:
    """Squared Euclidean distances matrix ||x-z||^2 using dot products."""
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    Xn = np.sum(X * X, axis=1, keepdims=True)  # (n,1)
    Zn = np.sum(Z * Z, axis=1, keepdims=True).T  # (1,m)
    return (Xn + Zn - 2.0 * (X @ Z.T)).astype(np.float32, copy=False)

# Gamma heuristic + grid
gamma0 = estimate_gamma(X_train, sample=min(500, X_train.shape[0]), seed=0)
gammas = [gamma0 / 3, gamma0, gamma0 * 3]
lams = [1e-3, 3e-3, 1e-2]

print("Precomputing squared distances...")
D2_tr = sq_dists(X_train, X_train)
D2_val = sq_dists(X_val, X_train)
Y_tr = one_hot(y_train, n_classes)
I = np.eye(D2_tr.shape[0], dtype=np.float32)

best = {"acc": -1.0}
for gamma, lam in product(gammas, lams):
    K_tr = np.exp(-float(gamma) * D2_tr).astype(np.float32, copy=False)
    A = np.linalg.solve(K_tr + float(lam) * I, Y_tr)
    K_val = np.exp(-float(gamma) * D2_val).astype(np.float32, copy=False)
    pred = (K_val @ A).argmax(axis=1)
    acc = float(np.mean(pred == y_val))
    print(f"lam={lam:.1e} gamma={gamma:.3e} val_acc={acc:.4f}")
    if acc > best["acc"]:
        best = {"acc": acc, "lam": float(lam), "gamma": float(gamma)}

print("Best RBF-KRR params:", best)

print("Training final RBF-KRR on subset...")
D2_sub = sq_dists(X_sub, X_sub)
K_sub = np.exp(-best["gamma"] * D2_sub).astype(np.float32, copy=False)
Y_sub = one_hot(y_sub, n_classes)
A_sub = np.linalg.solve(K_sub + best["lam"] * np.eye(K_sub.shape[0], dtype=np.float32), Y_sub)

print("Computing test cross-distances...")
D2_test = sq_dists(X_test_hog_s_rbf.astype(np.float32, copy=False), X_sub)
K_test = np.exp(-best["gamma"] * D2_test).astype(np.float32, copy=False)
scores_test = K_test @ A_sub
yte_int = scores_test.argmax(axis=1)
yte = np.array([inv_map_rbf[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_hog_krr_rbf.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## HOG + SVM with Chi-square kernel
**Why Chi-square?** HOG features behave like *histograms of oriented gradients* aggregated over cells/blocks. For histogram-like, nonnegative features, the Chi-square distance often compares bins better than Euclidean distance.

### Chi-square distance and kernel
For two nonnegative feature vectors $x,z\in\mathbb{R}^d_{\ge 0}$, the (symmetric) Chi-square distance is:
$$
\chi^2(x,z)=\sum_{j=1}^d \frac{(x_j-z_j)^2}{x_j+z_j+\varepsilon}.
$$
A common positive definite kernel built from it is an RBF-style kernel:
$$
k_{\chi^2}(x,z)=\exp\big(-\gamma\,\chi^2(x,z)\big),\qquad \gamma>0.
$$

### Practical notes
- Ensure features are **nonnegative** and usually **$\ell_1$-normalised** (so each vector sums to 1) before computing $\chi^2$.
- Training a kernel SVM needs Gram matrices of size $n\times n$; for speed/memory we often train on a **subset** and predict the test set with cross-kernels.

In [ ]:
# SVM (precomputed kernel) on HOG + RBF–Chi² kernel
# Goal: small/fast tuning grid, then train best model on a larger subset.

import numpy as np
import pandas as pd

from src.data import load_data, encode_labels
from src.features import compute_hog_features

# -----------------------------
# Helpers
# -----------------------------

def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)


def compute_hog_variant(X_raw: np.ndarray, spec: dict) -> np.ndarray:
    """Compute a HOG feature variant; supports single config or concatenation of two configs."""
    if "concat" in spec:
        feats = []
        for params in spec["concat"]:
            feats.append(compute_hog_features(X_raw, **params))
        X = np.concatenate(feats, axis=1)
    else:
        X = compute_hog_features(X_raw, **spec)
    return l1_normalize_nonneg(X)


def stratified_split_idx(y: np.ndarray, test_size: float, seed: int):
    y = np.asarray(y)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=float(test_size), random_state=int(seed))
    tr_idx, va_idx = next(sss.split(np.zeros_like(y), y))
    return tr_idx, va_idx


def chi2_dist_from_kernel(K: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """If K = exp(-D) (gamma=1), recover D = -log(K)."""
    K = np.asarray(K)
    return (-np.log(np.clip(K, eps, 1.0))).astype(np.float32, copy=False)


def estimate_gamma_chi2(X: np.ndarray, sample: int = 500, seed: int = 0) -> float:
    """Median-heuristic gamma for exp(-gamma * chi2(x,z))."""
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]

    # chi2_kernel with gamma=1 returns exp(-D)
    K = chi2_kernel(Xs, Xs, gamma=1.0)
    D = chi2_dist_from_kernel(K)
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))


def tune_gamma_C_from_dist(
    D_tr: np.ndarray,
    D_va: np.ndarray,
    y_tr: np.ndarray,
    y_va: np.ndarray,
    *,
    gamma0: float,
    Cs: list[float],
    gamma_factors: list[float],
) -> dict:
    best = {"acc": -1.0}
    for gf in gamma_factors:
        gamma = float(gamma0 * gf)
        K_tr = np.exp(-gamma * D_tr).astype(np.float32, copy=False)
        K_va = np.exp(-gamma * D_va).astype(np.float32, copy=False)
        for C in Cs:
            clf = SVC(C=float(C), kernel="precomputed", cache_size=2000)
            clf.fit(K_tr, y_tr)
            acc = float(np.mean(clf.predict(K_va) == y_va))
            if acc > best["acc"]:
                best = {"acc": acc, "C": float(C), "gamma": float(gamma)}
    return best


# -----------------------------
# 1) Load data
# -----------------------------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

# -----------------------------
# 2) Fast tuning on a subset (few HOG configs, small (gamma,C) grids)
# -----------------------------
seed = 42
val_size = 0.2
subset_size_tune = 1500  # reduced grid-search budget

X_tune_raw = X_raw[:subset_size_tune]
y_tune = y_int[:subset_size_tune]
tr_idx, va_idx = stratified_split_idx(y_tune, test_size=val_size, seed=seed)

# Keep only a few strong candidates (grid reduced)
hog_grid = [
    dict(name="hog_9_4x4_3x3", orientations=9, pixels_per_cell=(4, 4), cells_per_block=(3, 3)),
    dict(name="hog_12_6x6_3x3", orientations=12, pixels_per_cell=(6, 6), cells_per_block=(3, 3)),
    dict(
        name="hog_concat_12_6x6_3x3__plus__9_4x4_2x2",
        concat=[
            dict(orientations=12, pixels_per_cell=(6, 6), cells_per_block=(3, 3)),
            dict(orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2)),
        ],
    ),
]

# Small grids (coarse + tiny refine)
Cs_coarse = [0.3, 1.0, 3.0, 10.0]
gamma_factors_coarse = [0.5, 1.0, 2.0]
Cs_refine_factor = [1 / 3, 1.0, 3.0]
gamma_factors_refine = [1 / 3, 1.0, 3.0]

best_overall = {"acc": -1.0}

for i, spec in enumerate(hog_grid, start=1):
    name = spec.get("name", f"hog_{i}")
    spec_params = {k: v for k, v in spec.items() if k not in ("name",)}

    print("=" * 70)
    print(f"HOG config {i}/{len(hog_grid)}: {name}")

    X_hog = compute_hog_variant(X_tune_raw, spec_params)
    print(f"  feature_dim={int(X_hog.shape[1])}")

    X_tr = X_hog[tr_idx]
    X_va = X_hog[va_idx]
    y_tr = y_tune[tr_idx]
    y_va = y_tune[va_idx]

    # Compute chi2 distances once (via chi2_kernel gamma=1), then re-use for multiple gammas.
    print("  Precomputing Chi2 distances (optimized)...")
    D_tr = chi2_dist_from_kernel(chi2_kernel(X_tr, X_tr, gamma=1.0))
    D_va = chi2_dist_from_kernel(chi2_kernel(X_va, X_tr, gamma=1.0))

    gamma0 = estimate_gamma_chi2(X_tr, sample=min(500, X_tr.shape[0]), seed=seed)
    print(f"  gamma0={gamma0:.3e}")

    best1 = tune_gamma_C_from_dist(
        D_tr,
        D_va,
        y_tr,
        y_va,
        gamma0=gamma0,
        Cs=Cs_coarse,
        gamma_factors=gamma_factors_coarse,
    )
    print("   best coarse:", best1)

    Cs_ref = sorted({float(best1["C"] * f) for f in Cs_refine_factor if best1["C"] * f > 0})
    best2 = tune_gamma_C_from_dist(
        D_tr,
        D_va,
        y_tr,
        y_va,
        gamma0=float(best1["gamma"]),
        Cs=Cs_ref,
        gamma_factors=gamma_factors_refine,
    )
    print("   best refine:", best2)

    if best2["acc"] > best_overall["acc"]:
        best_overall = {
            **best2,
            "hog_spec": spec,
            "feature_dim": int(X_hog.shape[1]),
        }
        print("  -> New best overall")

print("=" * 70)
print("Best HOG+RBF–Chi2 on tuning subset:", best_overall)

# -----------------------------
# 3) Train final model longer (bigger subset), using the tuned hyperparams
# -----------------------------
subset_size_final = int(min(5000, X_raw.shape[0]))  # increase for better final fit

spec = best_overall["hog_spec"]
name = spec.get("name", "best_hog")
spec_params = {k: v for k, v in spec.items() if k not in ("name",)}

X_sub_raw = X_raw[:subset_size_final]
y_sub = y_int[:subset_size_final]

print("\nComputing HOG for final training subset + test...")
print(f"  subset_size_final={subset_size_final}")
print(f"  best spec: {name}")

X_sub_hog = compute_hog_variant(X_sub_raw, spec_params)
X_test_hog = compute_hog_variant(X_test_raw, spec_params)
print(f"  feature_dim={int(X_sub_hog.shape[1])}")

best_gamma = float(best_overall["gamma"])
best_C = float(best_overall["C"])

print("Precomputing final Gram matrix (optimized)...")
K_sub = chi2_kernel(X_sub_hog, X_sub_hog, gamma=best_gamma).astype(np.float32, copy=False)
final_clf = SVC(C=best_C, kernel="precomputed", cache_size=4000)
final_clf.fit(K_sub, y_sub)

print("Computing test cross-kernel...")
K_test = chi2_kernel(X_test_hog, X_sub_hog, gamma=best_gamma).astype(np.float32, copy=False)
yte_int = final_clf.predict(K_test)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_hog_svm_chi2_tuned_featuresize.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(
    {
        "subset_size_final": subset_size_final,
        "hog_spec": best_overall["hog_spec"],
        "feature_dim": int(X_sub_hog.shape[1]),
        "acc_tune": float(best_overall["acc"]),
        "C": best_C,
        "gamma": best_gamma,
    }
)

## Scattering transform + SVM (RBF kernel)
Le **scattering transform** (Mallat) extrait des représentations invariantes (petites translations) et stables aux déformations, en cascades de convolutions par ondelettes + module + moyennage.

### Intuition
- On applique des filtres ondelettes à plusieurs échelles/orientations.
- On prend le **module** (non-linéarité) puis on ré-applique des ondelettes (ordre 1, ordre 2, …).
- On obtient un descripteur riche mais plus stable qu’un CNN “appris” quand on a peu de tuning.

Ensuite on entraîne un **SVM à noyau RBF** sur ces features :
$$k(x,z)=\exp(-\gamma\,\|x-z\|_2^2).$$
Ici $x,z$ sont des vecteurs de scattering (après flatten + standardisation).

> Note: le SVM RBF reste coûteux en $O(n^2)$, donc on entraîne sur un **subset**.

In [ ]:
import numpy as np
import pandas as pd


from src.data import load_data, encode_labels, train_val_split

def to_gray_images(X_flat: np.ndarray) -> np.ndarray:
    """(n, 3072) -> (n, 32, 32) float32 in [0,1]."""
    X = np.asarray(X_flat, dtype=np.float32)
    X = X.reshape(-1, 3, 32, 32)  # (n, c, h, w)
    Xg = X.mean(axis=1)  # grayscale (n, 32, 32)
    mx = float(np.max(Xg))
    if mx > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0)
    return Xg.astype(np.float32, copy=False)

def scattering_features(
    X_img: np.ndarray,
    *,
    J: int = 2,
    L: int = 8,
    batch_size: int = 256,
) -> np.ndarray:
    """Compute flattened scattering features with batching."""
    scat = Scattering2D(J=J, shape=(32, 32), L=L)
    n = X_img.shape[0]
    feats = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(X_img[i0:i1])  # (b, C, H', W')
        feats.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    return np.vstack(feats)

DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

print("Preparing grayscale images...")
X_img = to_gray_images(X_raw)
X_test_img = to_gray_images(X_test_raw)

# Subset for speed (RBF SVM is O(n^2))
subset_size = 1800
X_sub_img = X_img[:subset_size]
y_sub = y_int[:subset_size]

print("Computing scattering features (subset/train/val/test)...")
X_train_img, X_val_img, y_train, y_val = train_val_split(X_sub_img, y_sub, test_size=0.2, seed=42)

X_train_scat = scattering_features(X_train_img, J=2, L=8, batch_size=256)
X_val_scat = scattering_features(X_val_img, J=2, L=8, batch_size=256)
X_sub_scat = scattering_features(X_sub_img, J=2, L=8, batch_size=256)
X_test_scat = scattering_features(X_test_img, J=2, L=8, batch_size=256)

# Standardize (important for RBF SVM)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scat_s = scaler.fit_transform(X_train_scat)
X_val_scat_s = scaler.transform(X_val_scat)
X_sub_scat_s = scaler.transform(X_sub_scat)
X_test_scat_s = scaler.transform(X_test_scat)

Cs = [1.0, 3.0, 10.0]
gammas = ["scale", "auto"]  # keep it small
best = {"acc": -1.0}
for C in Cs:
    for gamma in gammas:
        clf = SVC(C=float(C), kernel="rbf", gamma=gamma)
        clf.fit(X_train_scat_s, y_train)
        pred = clf.predict(X_val_scat_s)
        acc = float(np.mean(pred == y_val))
        print(f"C={C:.3g} gamma={gamma} val_acc={acc:.4f}")
        if acc > best["acc"]:
            best = {"acc": acc, "C": float(C), "gamma": gamma}

print("Best params:", best)

print("Training final RBF-SVM on subset...")
final_clf = SVC(C=best["C"], kernel="rbf", gamma=best["gamma"])
final_clf.fit(X_sub_scat_s, y_sub)
yte_int = final_clf.predict(X_test_scat_s)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_scattering_svm_rbf.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## Scattering2D + SVM (RBF–Chi² kernel)
On garde les **features de scattering** mais on remplace la distance Euclidienne par une distance **Chi-square** (adaptée aux features non négatives).
On utilise alors un noyau RBF–Chi² :
$$k_{\chi^2}(x,z)=\exp\big(-\gamma\,\chi^2(x,z)\big),\qquad \chi^2(x,z)=\sum_j \frac{(x_j-z_j)^2}{x_j+z_j+\varepsilon}.$$
Comme pour tout SVM à noyau, on entraîne sur un **subset** et on prédit le test via un noyau croisé.

In [ ]:
# Scattering2D (Kymatio) + SVM (precomputed RBF–Chi² kernel) — subset
import numpy as np
import pandas as pd

from src.data import load_data, encode_labels, train_val_split

def to_gray_images(X_flat: np.ndarray) -> np.ndarray:
    """(n, 3072) -> (n, 32, 32) float32 in [0,1]."""
    X = np.asarray(X_flat, dtype=np.float32)
    X = X.reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    mx = float(np.max(Xg))
    if mx > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0)
    return Xg.astype(np.float32, copy=False)

def scattering_features(
    X_img: np.ndarray,
    *,
    J: int = 2,
    L: int = 8,
    batch_size: int = 256,
 ) -> np.ndarray:
    """Compute flattened scattering features with batching."""
    scat = Scattering2D(J=J, shape=(32, 32), L=L)
    n = X_img.shape[0]
    feats = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(X_img[i0:i1])  # (b, C, H', W')
        feats.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    return np.vstack(feats)

def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)

def chi2_dist_block(Xb: np.ndarray, Zb: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Chi-square distance block for nonnegative arrays."""
    X3 = Xb[:, None, :]
    Z3 = Zb[None, :, :]
    num = (X3 - Z3) ** 2
    den = X3 + Z3 + eps
    return np.sum(num / den, axis=2, dtype=np.float32)

def chi2_dist_matrix(
    X: np.ndarray,
    Z: np.ndarray,
    *,
    eps: float = 1e-12,
    chunk_x: int = 32,
    chunk_z: int = 256,
 ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n, m = X.shape[0], Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_x):
        i1 = min(i0 + chunk_x, n)
        Xb = X[i0:i1]
        for j0 in range(0, m, chunk_z):
            j1 = min(j0 + chunk_z, m)
            Zb = Z[j0:j1]
            D[i0:i1, j0:j1] = chi2_dist_block(Xb, Zb, eps=eps)
    return D

def estimate_gamma_chi2(X: np.ndarray, sample: int = 120, seed: int = 0) -> float:
    """Median-heuristic gamma for exp(-gamma * chi2(x,z))."""
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D = np.empty((s, s), dtype=np.float32)
    for i in range(s):
        D[i] = chi2_dist_block(Xs[i : i + 1], Xs)[0]
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))

def rbf_chi2_kernel_from_dist(D: np.ndarray, gamma: float) -> np.ndarray:
    return np.exp(-float(gamma) * np.asarray(D, dtype=np.float32)).astype(np.float32, copy=False)

def _unique_sorted(vals) -> list[float]:
    return sorted({float(v) for v in vals})

DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

print("Preparing grayscale images...")
X_img = to_gray_images(X_raw)
X_test_img = to_gray_images(X_test_raw)

# Subset for speed (kernel SVM is O(n^2))
subset_size = 1200
X_sub_img = X_img[:subset_size]
y_sub = y_int[:subset_size]

print("Computing scattering features (subset/train/val/test)...")
X_train_img, X_val_img, y_train, y_val = train_val_split(
    X_sub_img, y_sub, test_size=0.2, seed=42
 )

X_train_scat = scattering_features(X_train_img, J=2, L=8, batch_size=256)
X_val_scat = scattering_features(X_val_img, J=2, L=8, batch_size=256)
X_sub_scat = scattering_features(X_sub_img, J=2, L=8, batch_size=256)
X_test_scat = scattering_features(X_test_img, J=2, L=8, batch_size=256)

# Chi² kernel expects nonnegative vectors; apply safe nonneg + L1-normalization
X_train_scat = l1_normalize_nonneg(X_train_scat)
X_val_scat = l1_normalize_nonneg(X_val_scat)
X_sub_scat = l1_normalize_nonneg(X_sub_scat)
X_test_scat = l1_normalize_nonneg(X_test_scat)

print("Precomputing Chi² distances...")
D_tr = chi2_dist_matrix(X_train_scat, X_train_scat, chunk_x=32, chunk_z=256)
D_val = chi2_dist_matrix(X_val_scat, X_train_scat, chunk_x=32, chunk_z=256)

# -----------------------------
# 5-min tuning: coarse -> refine grid on (gamma, C)
# -----------------------------
gamma0 = estimate_gamma_chi2(X_train_scat, sample=min(150, X_train_scat.shape[0]), seed=0)
print(f"gamma0 (median heuristic) = {gamma0:.3e}")

gammas_coarse = gamma0 * np.array([1 / 10, 1 / 3, 1.0, 3.0, 10.0], dtype=np.float32)
Cs_coarse = [0.3, 1.0, 3.0, 10.0]

best = {"acc": -1.0, "C": float(Cs_coarse[0]), "gamma": float(gammas_coarse[0])}

def run_grid(gammas, Cs, *, tag: str) -> None:
    print(f"--- {tag} ---")
    for gamma in gammas:
        K_tr = rbf_chi2_kernel_from_dist(D_tr, gamma)
        K_val = rbf_chi2_kernel_from_dist(D_val, gamma)
        for C in Cs:
            clf = SVC(C=float(C), kernel="precomputed")
            clf.fit(K_tr, y_train)
            pred = clf.predict(K_val)
            acc = float(np.mean(pred == y_val))
            print(f"C={float(C):.3g} gamma={float(gamma):.3e} val_acc={acc:.4f}")
            if acc > best["acc"]:
                best.update({"acc": acc, "C": float(C), "gamma": float(gamma)})

run_grid(gammas_coarse, Cs_coarse, tag="coarse grid")

# Refine locally around the coarse best (small extra cost)
gammas_refine = _unique_sorted([best["gamma"] / 3.0, best["gamma"], best["gamma"] * 3.0])
Cs_refine = _unique_sorted([best["C"] / 3.0, best["C"], best["C"] * 3.0])
run_grid(gammas_refine, Cs_refine, tag="refine grid")

print("Best params:", best)

print("Training final SVM on subset...")
D_sub = chi2_dist_matrix(X_sub_scat, X_sub_scat, chunk_x=32, chunk_z=256)
K_sub = rbf_chi2_kernel_from_dist(D_sub, best["gamma"])
final_clf = SVC(C=best["C"], kernel="precomputed")
final_clf.fit(K_sub, y_sub)

print("Computing test cross-kernel...")
D_test = chi2_dist_matrix(X_test_scat, X_sub_scat, chunk_x=32, chunk_z=256)
K_test = rbf_chi2_kernel_from_dist(D_test, best["gamma"])
yte_int = final_clf.predict(K_test)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_scattering_svm_rbf_chi2.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## HOPG + SVM (noyau Chi-square)

Ici on utilise **HOPG (Histogram of Oriented Phase Gradients)** au lieu de HOG.
L’idée :
- On calcule une **carte de phase locale** (via réponses complexes de filtres de Gabor).
- On prend le **gradient spatial de la phase** (après *unwrap*), ce qui met l’accent sur la structure et est moins sensible aux variations d’illumination que les gradients d’intensité.
- On construit un histogramme d’orientations (comme HOG) en pondérant par la magnitude du gradient de phase.

Ensuite, comme ce sont des features de type histogramme (non négatives après pondération), on utilise un **noyau Chi-square** :
$$k_{\chi^2}(x,z)=\exp\big(-\gamma\,\chi^2(x,z)\big),\qquad \chi^2(x,z)=\sum_j \frac{(x_j-z_j)^2}{x_j+z_j+\varepsilon}.$$

> Remarque : un SVM à noyau est coûteux en $O(n^2)$, donc on entraîne sur un subset.

In [ ]:
# HOPG (Histogram of Oriented Phase Gradients) + SVM (precomputed Chi-square kernel)
import numpy as np
import pandas as pd

from src.data import load_data, encode_labels, train_val_split

# -----------------------------
# HOPG feature extraction
# -----------------------------
def _phase_gradient_field(
    img2d: np.ndarray,
    *,
    gabor_orientations: int = 6,
    frequencies: tuple[float, ...] = (0.2, 0.35),
    eps: float = 1e-12,
    seed: int = 0,
 ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute a robust phase-gradient field by selecting the Gabor response with max amplitude per pixel.

    Returns: gx, gy, mag (all float32, shape (H,W)).
    """
    img = np.asarray(img2d, dtype=np.float32)
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    H, W = img.shape
    thetas = np.linspace(0.0, np.pi, int(gabor_orientations), endpoint=False)
    rng = np.random.default_rng(seed)
    # Break ties deterministically but avoid all-zero weirdness
    mag_max = np.full((H, W), -np.inf, dtype=np.float32)
    gx_best = np.zeros((H, W), dtype=np.float32)
    gy_best = np.zeros((H, W), dtype=np.float32)

    for theta in thetas:
        for freq in frequencies:
            real, imag = gabor(img, frequency=float(freq), theta=float(theta))
            phase = np.arctan2(imag, real).astype(np.float32, copy=False)
            # Unwrap phase along both axes to reduce discontinuities
            phase_u = np.unwrap(np.unwrap(phase, axis=0), axis=1).astype(np.float32, copy=False)
            gy, gx = np.gradient(phase_u)
            gx = gx.astype(np.float32, copy=False)
            gy = gy.astype(np.float32, copy=False)
            mag = np.sqrt(gx * gx + gy * gy + eps).astype(np.float32, copy=False)
            # Choose the strongest phase-gradient per pixel
            mask = mag > mag_max
            if np.any(mask):
                mag_max[mask] = mag[mask]
                gx_best[mask] = gx[mask]
                gy_best[mask] = gy[mask]

    mag_max = np.maximum(mag_max, 0.0).astype(np.float32, copy=False)
    return gx_best, gy_best, mag_max


def compute_hopg_features(
    X_flat: np.ndarray,
    *,
    orientations: int = 9,
    pixels_per_cell: tuple[int, int] = (4, 4),
    cells_per_block: tuple[int, int] = (2, 2),
    gabor_orientations: int = 6,
    frequencies: tuple[float, ...] = (0.2, 0.35),
 ) -> np.ndarray:
    """Compute HOPG features for flattened 32x32x3 images.

    Output is nonnegative (histograms) and float32.
    """
    X = np.asarray(X_flat, dtype=np.float32)
    X = X.reshape(-1, 3, 32, 32)
    # grayscale
    Xg = X.mean(axis=1)
    if float(np.nanmax(Xg)) > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0).astype(np.float32, copy=False)

    n = Xg.shape[0]
    cy, cx = pixels_per_cell
    by, bx = cells_per_block
    n_cells_y = 32 // cy
    n_cells_x = 32 // cx
    n_bins = int(orientations)
    bin_edges = np.linspace(0.0, 2.0 * np.pi, n_bins + 1, endpoint=True).astype(np.float32)

    # per-cell histograms: (n, n_cells_y, n_cells_x, n_bins)
    cell_hist = np.zeros((n, n_cells_y, n_cells_x, n_bins), dtype=np.float32)

    for i in range(n):
        gx, gy, mag = _phase_gradient_field(
            Xg[i],
            gabor_orientations=gabor_orientations,
            frequencies=frequencies,
            seed=i,
        )
        ang = (np.arctan2(gy, gx) % (2.0 * np.pi)).astype(np.float32, copy=False)

        # Accumulate histogram per cell (simple binning, weighted by mag)
        for yy in range(n_cells_y):
            y0, y1 = yy * cy, (yy + 1) * cy
            for xx in range(n_cells_x):
                x0, x1 = xx * cx, (xx + 1) * cx
                a = ang[y0:y1, x0:x1].ravel()
                w = mag[y0:y1, x0:x1].ravel()
                # bin index in [0, n_bins-1]
                b = np.clip(np.searchsorted(bin_edges, a, side="right") - 1, 0, n_bins - 1)
                hist = np.bincount(b, weights=w, minlength=n_bins).astype(np.float32, copy=False)
                cell_hist[i, yy, xx] = hist

    # Block normalization like HOG (L2) to improve robustness
    n_blocks_y = n_cells_y - by + 1
    n_blocks_x = n_cells_x - bx + 1
    eps = 1e-6
    blocks = []
    for yy in range(n_blocks_y):
        for xx in range(n_blocks_x):
            block = cell_hist[:, yy : yy + by, xx : xx + bx, :].reshape(n, -1)
            norm = np.sqrt(np.sum(block * block, axis=1, keepdims=True) + eps)
            blocks.append(block / norm)
    feats = np.concatenate(blocks, axis=1).astype(np.float32, copy=False)
    feats = np.maximum(feats, 0.0)
    return feats


def l1_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)

# -----------------------------
# Chi-square kernel (precomputed SVM)
# -----------------------------
def chi2_dist_block(Xb: np.ndarray, Zb: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X3 = Xb[:, None, :]
    Z3 = Zb[None, :, :]
    num = (X3 - Z3) ** 2
    den = X3 + Z3 + eps
    return np.sum(num / den, axis=2, dtype=np.float32)

def chi2_dist_matrix(
    X: np.ndarray,
    Z: np.ndarray,
    *,
    eps: float = 1e-12,
    chunk_x: int = 32,
    chunk_z: int = 256,
 ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n, m = X.shape[0], Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_x):
        i1 = min(i0 + chunk_x, n)
        Xb = X[i0:i1]
        for j0 in range(0, m, chunk_z):
            j1 = min(j0 + chunk_z, m)
            Zb = Z[j0:j1]
            D[i0:i1, j0:j1] = chi2_dist_block(Xb, Zb, eps=eps)
    return D

def estimate_gamma_chi2(X: np.ndarray, sample: int = 120, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D = np.empty((s, s), dtype=np.float32)
    for i in range(s):
        D[i] = chi2_dist_block(Xs[i : i + 1], Xs)[0]
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))

# -----------------------------
# Run pipeline
# -----------------------------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

subset_size = 2000  # kernel methods are O(n^2)
X_sub_raw = X_raw[:subset_size]
y_sub = y_int[:subset_size]

print("Computing HOPG features...")
X_sub_feat = compute_hopg_features(
    X_sub_raw,
    orientations=9,
    pixels_per_cell=(4, 4),
    cells_per_block=(2, 2),
    gabor_orientations=6,
    frequencies=(0.2, 0.35),
 )
X_test_feat = compute_hopg_features(
    X_test_raw,
    orientations=9,
    pixels_per_cell=(4, 4),
    cells_per_block=(2, 2),
    gabor_orientations=6,
    frequencies=(0.2, 0.35),
 )

# Chi2 kernel likes nonnegative, L1-normalized histograms
X_sub_feat = l1_normalize(X_sub_feat)
X_test_feat = l1_normalize(X_test_feat)

X_train, X_val, y_train, y_val = train_val_split(X_sub_feat, y_sub, test_size=0.2, seed=42)
print(f"Subset size={subset_size} -> train={X_train.shape[0]} val={X_val.shape[0]}")

print("Precomputing Chi2 distances...")
D_tr = chi2_dist_matrix(X_train, X_train, chunk_x=32, chunk_z=256)
D_val = chi2_dist_matrix(X_val, X_train, chunk_x=32, chunk_z=256)

gamma0 = estimate_gamma_chi2(X_train, sample=min(120, X_train.shape[0]), seed=0)
gammas = [gamma0 / 3.0, gamma0, gamma0 * 3.0]
Cs = [0.3, 1.0, 3.0, 10.0]

best = {"acc": -1.0}
for gamma in gammas:
    K_tr = np.exp(-float(gamma) * D_tr).astype(np.float32, copy=False)
    K_val = np.exp(-float(gamma) * D_val).astype(np.float32, copy=False)
    for C in Cs:
        clf = SVC(C=float(C), kernel="precomputed")
        clf.fit(K_tr, y_train)
        pred = clf.predict(K_val)
        acc = float(np.mean(pred == y_val))
        print(f"C={C:.3g} gamma={gamma:.3e} val_acc={acc:.4f}")
        if acc > best["acc"]:
            best = {"acc": acc, "C": float(C), "gamma": float(gamma)}

print("Best params:", best)

print("Training final SVM on subset...")
D_sub = chi2_dist_matrix(X_sub_feat, X_sub_feat, chunk_x=32, chunk_z=256)
K_sub = np.exp(-best["gamma"] * D_sub).astype(np.float32, copy=False)
final_clf = SVC(C=best["C"], kernel="precomputed")
final_clf.fit(K_sub, y_sub)

print("Computing test cross-kernel...")
D_test = chi2_dist_matrix(X_test_feat, X_sub_feat, chunk_x=32, chunk_z=256)
K_test = np.exp(-best["gamma"] * D_test).astype(np.float32, copy=False)
yte_int = final_clf.predict(K_test)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_hopg_svm_chi2.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print(f"Trained on subset_size={subset_size}")

## Multiple Feature Fusion + Multi-Kernel Learning (MKL) — version simple (weighted sum)

On combine plusieurs descripteurs avec plusieurs noyaux, puis on entraîne un SVM sur le noyau fusionné.

### Features / kernels utilisés ici
- **Scattering** (grayscale) → **RBF** : $k_s(x,z)=\exp(-\gamma_s\|x-z\|_2^2)$
- **HOPG** → **Chi-square** : $k_h(x,z)=\exp(-\gamma_h\,\chi^2(x,z))$
- **LBP** → **Chi-square** : $k_l(x,z)=\exp(-\gamma_l\,\chi^2(x,z))$

### Fusion (MKL simple)
On construit un noyau combiné :
$$K = w_s K_s + w_h K_h + w_l K_l, \qquad w_s,w_h,w_l\ge0,\ \ w_s+w_h+w_l=1.$$
On choisit $(w_s,w_h,w_l)$ et $C$ par validation sur un subset (grille de poids), puis on ré-entraine sur le subset final et on prédit le test via noyaux croisés.

> C’est une forme de MKL “basique” (late fusion). Un vrai MKL apprend les poids de manière continue, mais ce baseline est déjà très fort en vision.

In [ ]:
# Feature fusion (Scattering RBF) + (HOPG Chi2) + (LBP Chi2) with weighted-sum MKL (simple)
import numpy as np
import pandas as pd

from src.data import load_data, encode_labels, train_val_split

# -----------------------------
# Utils
# -----------------------------
def sanitize_finite(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    if not np.isfinite(X).all():
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
    return X

def to_gray_32x32(X_flat: np.ndarray) -> np.ndarray:
    X = np.asarray(X_flat, dtype=np.float32).reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    mx = float(np.nanmax(Xg))
    if mx > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0)
    return sanitize_finite(Xg)

def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)

# -----------------------------
# LBP (histogram) features
# -----------------------------
def lbp_hist_features(
    Xg: np.ndarray,
    *,
    P: int = 8,
    R: int = 1,
    method: str = "uniform",
    grid: tuple[int, int] = (2, 2),
 ) -> np.ndarray:
    """LBP histogram with a small spatial grid (default 2x2)."""
    Xg = np.asarray(Xg, dtype=np.float32)
    n, H, W = Xg.shape
    gy, gx = int(grid[0]), int(grid[1])
    n_bins = P + 2 if method == "uniform" else (2 ** P)
    feats = []
    for i in range(n):
        lbp = local_binary_pattern(Xg[i], P=P, R=R, method=method).astype(np.int32, copy=False)
        blocks = []
        for by in range(gy):
            for bx in range(gx):
                y0, y1 = by * (H // gy), (by + 1) * (H // gy)
                x0, x1 = bx * (W // gx), (bx + 1) * (W // gx)
                patch = lbp[y0:y1, x0:x1].ravel()
                hist = np.bincount(patch, minlength=n_bins).astype(np.float32, copy=False)
                blocks.append(hist)
        feats.append(np.concatenate(blocks, axis=0))
    Xf = np.vstack(feats).astype(np.float32, copy=False)
    return l1_normalize_nonneg(Xf)

# -----------------------------
# HOPG (phase gradient) features
# -----------------------------
def _phase_grad_best_response(
    img: np.ndarray,
    *,
    gabor_orientations: int = 4,
    frequencies: tuple[float, ...] = (0.25,),
    eps: float = 1e-12,
 ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Select per-pixel strongest Gabor response, then compute gradient of unwrapped phase."""
    img = np.asarray(img, dtype=np.float32)
    H, W = img.shape
    thetas = np.linspace(0.0, np.pi, int(gabor_orientations), endpoint=False)
    best_mag = np.full((H, W), -np.inf, dtype=np.float32)
    best_gx = np.zeros((H, W), dtype=np.float32)
    best_gy = np.zeros((H, W), dtype=np.float32)

    for theta in thetas:
        for freq in frequencies:
            real, imag = gabor(img, frequency=float(freq), theta=float(theta))
            phase = np.arctan2(imag, real).astype(np.float32, copy=False)
            phase_u = np.unwrap(np.unwrap(phase, axis=0), axis=1).astype(np.float32, copy=False)
            gy, gx = np.gradient(phase_u)
            gx = gx.astype(np.float32, copy=False)
            gy = gy.astype(np.float32, copy=False)
            mag = np.sqrt(gx * gx + gy * gy + eps).astype(np.float32, copy=False)
            mask = mag > best_mag
            if np.any(mask):
                best_mag[mask] = mag[mask]
                best_gx[mask] = gx[mask]
                best_gy[mask] = gy[mask]

    best_mag = np.maximum(best_mag, 0.0).astype(np.float32, copy=False)
    return best_gx, best_gy, best_mag

def hopg_features(
    Xg: np.ndarray,
    *,
    orientations: int = 9,
    pixels_per_cell: tuple[int, int] = (8, 8),
    cells_per_block: tuple[int, int] = (2, 2),
    gabor_orientations: int = 4,
    frequencies: tuple[float, ...] = (0.25,),
 ) -> np.ndarray:
    """Simple HOPG: phase-gradient orientation histograms + block L2 norm."""
    Xg = np.asarray(Xg, dtype=np.float32)
    n, H, W = Xg.shape
    cy, cx = pixels_per_cell
    by, bx = cells_per_block
    n_cells_y = H // cy
    n_cells_x = W // cx
    n_bins = int(orientations)
    bin_edges = np.linspace(0.0, 2.0 * np.pi, n_bins + 1, endpoint=True).astype(np.float32)
    cell_hist = np.zeros((n, n_cells_y, n_cells_x, n_bins), dtype=np.float32)

    for i in range(n):
        gx, gy, mag = _phase_grad_best_response(
            Xg[i],
            gabor_orientations=gabor_orientations,
            frequencies=frequencies,
        )
        ang = (np.arctan2(gy, gx) % (2.0 * np.pi)).astype(np.float32, copy=False)
        for yy in range(n_cells_y):
            y0, y1 = yy * cy, (yy + 1) * cy
            for xx in range(n_cells_x):
                x0, x1 = xx * cx, (xx + 1) * cx
                a = ang[y0:y1, x0:x1].ravel()
                w = mag[y0:y1, x0:x1].ravel()
                b = np.clip(np.searchsorted(bin_edges, a, side="right") - 1, 0, n_bins - 1)
                cell_hist[i, yy, xx] = np.bincount(b, weights=w, minlength=n_bins).astype(np.float32, copy=False)

    # Block L2 normalization
    eps = 1e-6
    n_blocks_y = n_cells_y - by + 1
    n_blocks_x = n_cells_x - bx + 1
    blocks = []
    for yy in range(n_blocks_y):
        for xx in range(n_blocks_x):
            block = cell_hist[:, yy : yy + by, xx : xx + bx, :].reshape(n, -1)
            norm = np.sqrt(np.sum(block * block, axis=1, keepdims=True) + eps)
            blocks.append(block / norm)
    Xf = np.concatenate(blocks, axis=1).astype(np.float32, copy=False)
    return l1_normalize_nonneg(Xf)

# -----------------------------
# Scattering features (grayscale)
# -----------------------------
def scattering_features_gray(
    Xg: np.ndarray,
    *,
    J: int = 2,
    L: int = 8,
    max_order: int = 2,
    batch_size: int = 256,
 ) -> np.ndarray:
    Xg = np.asarray(Xg, dtype=np.float32)
    scat = Scattering2D(J=J, shape=(32, 32), L=L, max_order=max_order)
    n = Xg.shape[0]
    feats = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(Xg[i0:i1])  # (b, C, H', W')
        feats.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    Xf = np.vstack(feats)
    return sanitize_finite(Xf)

def preprocess_scattering_for_rbf(Xtr: np.ndarray, Xte: np.ndarray):
    # log compression is common on scattering
    Xtr = np.log1p(np.maximum(Xtr, 0.0)).astype(np.float32, copy=False)
    Xte = np.log1p(np.maximum(Xte, 0.0)).astype(np.float32, copy=False)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)
    # PCA to stabilize & reduce dims; keep variance ratio
    pca = PCA(n_components=0.98, whiten=True, random_state=42)
    Xtr_p = pca.fit_transform(Xtr_s)
    Xte_p = pca.transform(Xte_s)
    return sanitize_finite(Xtr_p), sanitize_finite(Xte_p)

# -----------------------------
# Kernel builders
# -----------------------------
def sq_dists(X: np.ndarray, Z: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    Xn = np.sum(X * X, axis=1, keepdims=True)
    Zn = np.sum(Z * Z, axis=1, keepdims=True).T
    return (Xn + Zn - 2.0 * (X @ Z.T)).astype(np.float32, copy=False)

def chi2_dist_block(Xb: np.ndarray, Zb: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X3 = Xb[:, None, :]
    Z3 = Zb[None, :, :]
    num = (X3 - Z3) ** 2
    den = X3 + Z3 + eps
    return np.sum(num / den, axis=2, dtype=np.float32)

def chi2_dist_matrix(X: np.ndarray, Z: np.ndarray, *, eps: float = 1e-12, chunk_x: int = 64, chunk_z: int = 512) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n, m = X.shape[0], Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_x):
        i1 = min(i0 + chunk_x, n)
        Xb = X[i0:i1]
        for j0 in range(0, m, chunk_z):
            j1 = min(j0 + chunk_z, m)
            Zb = Z[j0:j1]
            D[i0:i1, j0:j1] = chi2_dist_block(Xb, Zb, eps=eps)
    return D

def estimate_gamma_rbf(X: np.ndarray, sample: int = 250, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D2 = sq_dists(Xs, Xs)
    tri = D2[np.triu_indices_from(D2, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (2.0 * (med + 1e-12)))

def estimate_gamma_chi2(X: np.ndarray, sample: int = 120, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D = np.empty((s, s), dtype=np.float32)
    for i in range(s):
        D[i] = chi2_dist_block(Xs[i : i + 1], Xs)[0]
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))

# -----------------------------
# Load data
# -----------------------------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

# -----------------------------
# Subsets (keep MKL feasible)
# -----------------------------
# Tune on a smaller subset, then train/predict with a larger subset.
subset_tune = 1200
subset_final = 2000

X_tune_raw = X_raw[:subset_tune]
y_tune = y_int[:subset_tune]
X_final_raw = X_raw[:subset_final]
y_final = y_int[:subset_final]

# -----------------------------
# Compute features
# -----------------------------
print("Computing grayscale tensors...")
Xg_tune = to_gray_32x32(X_tune_raw)
Xg_final = to_gray_32x32(X_final_raw)
Xg_test = to_gray_32x32(X_test_raw)

print("Computing LBP features...")
X_lbp_tune = lbp_hist_features(Xg_tune, P=8, R=1, method="uniform", grid=(2, 2))
X_lbp_final = lbp_hist_features(Xg_final, P=8, R=1, method="uniform", grid=(2, 2))
X_lbp_test = lbp_hist_features(Xg_test, P=8, R=1, method="uniform", grid=(2, 2))

print("Computing HOPG features...")
X_hopg_tune = hopg_features(Xg_tune, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), gabor_orientations=4, frequencies=(0.25,))
X_hopg_final = hopg_features(Xg_final, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), gabor_orientations=4, frequencies=(0.25,))
X_hopg_test = hopg_features(Xg_test, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), gabor_orientations=4, frequencies=(0.25,))

print("Computing Scattering features...")
X_scat_tune = scattering_features_gray(Xg_tune, J=2, L=8, max_order=2, batch_size=256)
X_scat_final = scattering_features_gray(Xg_final, J=2, L=8, max_order=2, batch_size=256)
X_scat_test = scattering_features_gray(Xg_test, J=2, L=8, max_order=2, batch_size=256)
X_scat_tune_p, X_scat_test_p_tmp = preprocess_scattering_for_rbf(X_scat_tune, X_scat_test)
X_scat_final_p, X_scat_test_p = preprocess_scattering_for_rbf(X_scat_final, X_scat_test)

# -----------------------------
# Train/val split for tuning
# -----------------------------
X_idx = np.arange(subset_tune)
tr_idx, va_idx, y_tr, y_va = train_val_split(X_idx, y_tune, test_size=0.2, seed=42)
tr_idx = np.asarray(tr_idx)
va_idx = np.asarray(va_idx)
print(f"Tune split: train={tr_idx.size} val={va_idx.size}")

# Slice features according to indices
sc_tr, sc_va = X_scat_tune_p[tr_idx], X_scat_tune_p[va_idx]
ho_tr, ho_va = X_hopg_tune[tr_idx], X_hopg_tune[va_idx]
lb_tr, lb_va = X_lbp_tune[tr_idx], X_lbp_tune[va_idx]

# -----------------------------
# Build per-kernel matrices on tune split
# -----------------------------
print("Estimating gammas...")
gamma_s = estimate_gamma_rbf(sc_tr, sample=min(250, sc_tr.shape[0]), seed=0)
gamma_h = estimate_gamma_chi2(ho_tr, sample=min(120, ho_tr.shape[0]), seed=0)
gamma_l = estimate_gamma_chi2(lb_tr, sample=min(120, lb_tr.shape[0]), seed=0)
print({"gamma_scat": gamma_s, "gamma_hopg": gamma_h, "gamma_lbp": gamma_l})

print("Precomputing distance matrices...")
D2_sc_tr = sq_dists(sc_tr, sc_tr)
D2_sc_va = sq_dists(sc_va, sc_tr)
D_ho_tr = chi2_dist_matrix(ho_tr, ho_tr, chunk_x=64, chunk_z=512)
D_ho_va = chi2_dist_matrix(ho_va, ho_tr, chunk_x=64, chunk_z=512)
D_lb_tr = chi2_dist_matrix(lb_tr, lb_tr, chunk_x=64, chunk_z=512)
D_lb_va = chi2_dist_matrix(lb_va, lb_tr, chunk_x=64, chunk_z=512)

K_sc_tr = np.exp(-gamma_s * D2_sc_tr).astype(np.float32, copy=False)
K_sc_va = np.exp(-gamma_s * D2_sc_va).astype(np.float32, copy=False)
K_ho_tr = np.exp(-gamma_h * D_ho_tr).astype(np.float32, copy=False)
K_ho_va = np.exp(-gamma_h * D_ho_va).astype(np.float32, copy=False)
K_lb_tr = np.exp(-gamma_l * D_lb_tr).astype(np.float32, copy=False)
K_lb_va = np.exp(-gamma_l * D_lb_va).astype(np.float32, copy=False)

# -----------------------------
# MKL simple: grid over weights + C
# -----------------------------
def weight_grid(step: float = 0.25):
    ws = np.arange(0.0, 1.0 + 1e-9, step)
    out = []
    for w1 in ws:
        for w2 in ws:
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9:
                continue
            if w3 < 0.0:
                w3 = 0.0
            out.append((float(w1), float(w2), float(w3)))
    # unique (float) combos
    uniq = []
    seen = set()
    for w in out:
        key = (round(w[0], 6), round(w[1], 6), round(w[2], 6))
        if key not in seen:
            uniq.append(w)
            seen.add(key)
    return uniq

weights = weight_grid(step=0.25)
Cs = [0.3, 1.0, 3.0, 10.0]

best = {"acc": -1.0}
print(f"Tuning over {len(weights)} weight triplets and {len(Cs)} Cs...")
for (w_sc, w_ho, w_lb) in weights:
    K_tr = (w_sc * K_sc_tr + w_ho * K_ho_tr + w_lb * K_lb_tr).astype(np.float32, copy=False)
    K_va = (w_sc * K_sc_va + w_ho * K_ho_va + w_lb * K_lb_va).astype(np.float32, copy=False)
    for C in Cs:
        clf = SVC(C=float(C), kernel="precomputed")
        clf.fit(K_tr, y_tr)
        pred = clf.predict(K_va)
        acc = float(np.mean(pred == y_va))
        if acc > best["acc"]:
            best = {"acc": acc, "C": float(C), "w_sc": w_sc, "w_ho": w_ho, "w_lb": w_lb}
    print(f"weights(sc,hopg,lbp)=({w_sc:.2f},{w_ho:.2f},{w_lb:.2f}) | best_acc_so_far={best['acc']:.4f}")

print("Best tune:", best)

# -----------------------------
# Train final on subset_final and predict test
# -----------------------------
print("Building final kernels...")
w_sc, w_ho, w_lb = best["w_sc"], best["w_ho"], best["w_lb"]
C_final = best["C"]

# Use same gamma heuristics but re-estimate on final subset for stability
gamma_s_f = estimate_gamma_rbf(X_scat_final_p, sample=min(300, subset_final), seed=0)
gamma_h_f = estimate_gamma_chi2(X_hopg_final, sample=min(150, subset_final), seed=0)
gamma_l_f = estimate_gamma_chi2(X_lbp_final, sample=min(150, subset_final), seed=0)
print({"gamma_scat_final": gamma_s_f, "gamma_hopg_final": gamma_h_f, "gamma_lbp_final": gamma_l_f})

D2_sc_sub = sq_dists(X_scat_final_p, X_scat_final_p)
D2_sc_te = sq_dists(X_scat_test_p, X_scat_final_p)
D_ho_sub = chi2_dist_matrix(X_hopg_final, X_hopg_final, chunk_x=64, chunk_z=512)
D_ho_te = chi2_dist_matrix(X_hopg_test, X_hopg_final, chunk_x=64, chunk_z=512)
D_lb_sub = chi2_dist_matrix(X_lbp_final, X_lbp_final, chunk_x=64, chunk_z=512)
D_lb_te = chi2_dist_matrix(X_lbp_test, X_lbp_final, chunk_x=64, chunk_z=512)

K_sc_sub = np.exp(-gamma_s_f * D2_sc_sub).astype(np.float32, copy=False)
K_sc_te = np.exp(-gamma_s_f * D2_sc_te).astype(np.float32, copy=False)
K_ho_sub = np.exp(-gamma_h_f * D_ho_sub).astype(np.float32, copy=False)
K_ho_te = np.exp(-gamma_h_f * D_ho_te).astype(np.float32, copy=False)
K_lb_sub = np.exp(-gamma_l_f * D_lb_sub).astype(np.float32, copy=False)
K_lb_te = np.exp(-gamma_l_f * D_lb_te).astype(np.float32, copy=False)

K_sub = (w_sc * K_sc_sub + w_ho * K_ho_sub + w_lb * K_lb_sub).astype(np.float32, copy=False)
K_te = (w_sc * K_sc_te + w_ho * K_ho_te + w_lb * K_lb_te).astype(np.float32, copy=False)

print("Training final SVM on fused kernel...")
final_clf = SVC(C=float(C_final), kernel="precomputed")
final_clf.fit(K_sub, y_final)
yte_int = final_clf.predict(K_te)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_mkl_fusion.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print({"subset_tune": subset_tune, "subset_final": subset_final, "best": best})

In [ ]:
import numpy as np
import pandas as pd

from tqdm import tqdm

# =========================================================
# LOAD DATA
# =========================================================

DATA_DIR = "data/"

Xtr = pd.read_csv(DATA_DIR + "Xtr.csv", header=None)
Xte = pd.read_csv(DATA_DIR + "Xte.csv", header=None)

if Xtr.shape[1] == 3073:
    Xtr = Xtr.iloc[:, 1:]
if Xte.shape[1] == 3073:
    Xte = Xte.iloc[:, 1:]

Xtr = Xtr.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)
Xte = Xte.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)

# Replace missing/inf pixels early (prevents NaNs propagating to HOG/LBP)
Xtr = np.nan_to_num(Xtr, nan=0.0, posinf=0.0, neginf=0.0)
Xte = np.nan_to_num(Xte, nan=0.0, posinf=0.0, neginf=0.0)

Ytr = pd.read_csv(DATA_DIR + "Ytr.csv")
y = Ytr["Prediction"].to_numpy()

# =========================================================
# RESHAPE
# =========================================================

def to_rgb(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32).reshape(-1, 3, 32, 32)
    # scale to [0,1] if pixels look like 0..255
    if float(np.nanmax(X)) > 1.5:
        X = X / 255.0
    X = np.clip(X, 0.0, 1.0)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

Xtr_img = to_rgb(Xtr)
Xte_img = to_rgb(Xte)

# =========================================================
# HOG FEATURES
# =========================================================

def extract_hog(X: np.ndarray) -> np.ndarray:
    feats = []
    for img in tqdm(X, desc="HOG"):
        gray = img.mean(axis=0).astype(np.float32, copy=False)
        gray = np.nan_to_num(gray, nan=0.0, posinf=0.0, neginf=0.0)
        gray = np.clip(gray, 0.0, 1.0)
        f = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            feature_vector=True,
        )
        feats.append(f)
    Xf = np.asarray(feats, dtype=np.float32)
    Xf = np.nan_to_num(Xf, nan=0.0, posinf=0.0, neginf=0.0)
    # Chi2 features must be nonnegative
    return np.maximum(Xf, 0.0)

# =========================================================
# LBP FEATURES
# =========================================================

def extract_lbp(X: np.ndarray) -> np.ndarray:
    feats = []
    for img in tqdm(X, desc="LBP"):
        gray = img.mean(axis=0).astype(np.float32, copy=False)
        gray = np.nan_to_num(gray, nan=0.0, posinf=0.0, neginf=0.0)
        gray = np.clip(gray, 0.0, 1.0)
        lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
        hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
        hist = hist.astype(np.float32, copy=False)
        hist /= (float(hist.sum()) + 1e-8)
        feats.append(hist)
    Xf = np.asarray(feats, dtype=np.float32)
    Xf = np.nan_to_num(Xf, nan=0.0, posinf=0.0, neginf=0.0)
    return np.maximum(Xf, 0.0)

print("Extracting features...")

Xtr_hog = extract_hog(Xtr_img)
Xte_hog = extract_hog(Xte_img)

Xtr_lbp = extract_lbp(Xtr_img)
Xte_lbp = extract_lbp(Xte_img)

# =========================================================
# CHI2 KERNEL APPROXIMATION
# =========================================================

# Use separate samplers (their fitted feature-dimension differs)
chi2_hog = AdditiveChi2Sampler(sample_steps=2)
chi2_lbp = AdditiveChi2Sampler(sample_steps=2)

Xtr_hog_chi2 = chi2_hog.fit_transform(Xtr_hog)
Xte_hog_chi2 = chi2_hog.transform(Xte_hog)

Xtr_lbp_chi2 = chi2_lbp.fit_transform(Xtr_lbp)
Xte_lbp_chi2 = chi2_lbp.transform(Xte_lbp)

# =========================================================
# CONCATENATE FEATURES
# =========================================================

Xtr_final = np.hstack([Xtr_hog_chi2, Xtr_lbp_chi2])
Xte_final = np.hstack([Xte_hog_chi2, Xte_lbp_chi2])

scaler = StandardScaler()
Xtr_final = scaler.fit_transform(Xtr_final)
Xte_final = scaler.transform(Xte_final)

# =========================================================
# TRAIN LINEAR SVM
# =========================================================

clf = LinearSVC(C=5.0, class_weight="balanced", max_iter=5000)
clf.fit(Xtr_final, y)

y_pred_test = clf.predict(Xte_final)

# =========================================================
# SAVE
# =========================================================

submission = pd.DataFrame({
    "Id": np.arange(1, len(y_pred_test) + 1),
    "Prediction": y_pred_test,
})

submission.to_csv("results/submission_mkl_fast.csv", index=False)

print("Done.")

## LBP CHI 2 AND SVM


In [ ]:
import numpy as np
import pandas as pd

from tqdm import tqdm

# =========================================================
# LBP + Chi2 (AdditiveChi2Sampler) + SVM (LinearSVC)
# =========================================================

DATA_DIR = "data/"

# -------------------------
# Load data
# -------------------------
Xtr_df = pd.read_csv(DATA_DIR + "Xtr.csv", header=None)
Xte_df = pd.read_csv(DATA_DIR + "Xte.csv", header=None)

# Some provided files include an index column
if Xtr_df.shape[1] == 3073:
    Xtr_df = Xtr_df.iloc[:, 1:]
if Xte_df.shape[1] == 3073:
    Xte_df = Xte_df.iloc[:, 1:]

Xtr_raw = Xtr_df.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)
Xte_raw = Xte_df.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)

# Replace missing/inf pixels early
Xtr_raw = np.nan_to_num(Xtr_raw, nan=0.0, posinf=0.0, neginf=0.0)
Xte_raw = np.nan_to_num(Xte_raw, nan=0.0, posinf=0.0, neginf=0.0)

Ytr_df = pd.read_csv(DATA_DIR + "Ytr.csv")
y = Ytr_df["Prediction"].to_numpy()

def to_rgb(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32).reshape(-1, 3, 32, 32)
    if float(np.nanmax(X)) > 1.5:
        X = X / 255.0
    X = np.clip(X, 0.0, 1.0)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

Xtr_img = to_rgb(Xtr_raw)
Xte_img = to_rgb(Xte_raw)

# -------------------------
# LBP extractor
# -------------------------
def extract_lbp(X: np.ndarray, P: int = 8, R: int = 1) -> np.ndarray:
    # For uniform LBP, number of patterns is P+2 => 10 for P=8
    n_bins = P + 2
    feats = []
    for img in tqdm(X, desc="LBP"):
        gray = img.mean(axis=0).astype(np.float32, copy=False)
        gray = np.nan_to_num(gray, nan=0.0, posinf=0.0, neginf=0.0)
        gray = np.clip(gray, 0.0, 1.0)
        # Use uint8 to avoid skimage warning and improve numerical stability
        gray_u8 = np.clip((gray * 255.0).round(), 0.0, 255.0).astype(np.uint8, copy=False)
        lbp = local_binary_pattern(gray_u8, P=P, R=R, method="uniform")
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
        hist = hist.astype(np.float32, copy=False)
        hist = np.nan_to_num(hist, nan=0.0, posinf=0.0, neginf=0.0)
        hist /= (float(hist.sum()) + 1e-8)
        feats.append(hist)
    Xf = np.asarray(feats, dtype=np.float32)
    Xf = np.nan_to_num(Xf, nan=0.0, posinf=0.0, neginf=0.0)
    # Chi2 requires nonnegative features
    return np.maximum(Xf, 0.0)

print("Extracting LBP histograms...")
Xtr_lbp = extract_lbp(Xtr_img)
Xte_lbp = extract_lbp(Xte_img)

# -------------------------
# Chi2 feature map
# -------------------------
chi2 = AdditiveChi2Sampler(sample_steps=2)
Xtr_lbp_chi2 = chi2.fit_transform(Xtr_lbp)
Xte_lbp_chi2 = chi2.transform(Xte_lbp)

# Scaling helps LinearSVC
scaler = StandardScaler()
Xtr_final = scaler.fit_transform(Xtr_lbp_chi2)
Xte_final = scaler.transform(Xte_lbp_chi2)

# -------------------------
# Quick CV for C (optional but cheap)
# -------------------------
Cs = [0.3, 1.0, 3.0, 10.0]
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

best = {"acc": -1.0, "C": None}
for C in Cs:
    accs = []
    for tr_idx, va_idx in skf.split(Xtr_final, y):
        clf = LinearSVC(C=float(C), class_weight="balanced", max_iter=5000)
        clf.fit(Xtr_final[tr_idx], y[tr_idx])
        pred = clf.predict(Xtr_final[va_idx])
        accs.append(accuracy_score(y[va_idx], pred))
    mean_acc = float(np.mean(accs))
    if mean_acc > best["acc"]:
        best = {"acc": mean_acc, "C": float(C)}
    print(f"C={C} | cv_acc={mean_acc:.4f} | best={best}")

print("Training final model...")
final_clf = LinearSVC(C=float(best["C"]), class_weight="balanced", max_iter=5000)
final_clf.fit(Xtr_final, y)
y_pred_test = final_clf.predict(Xte_final)

out_path = "results/submission_lbp_chi2_svm.csv"
submission = pd.DataFrame({
    "Id": np.arange(1, len(y_pred_test) + 1),
    "Prediction": y_pred_test,
})
submission.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

In [ ]:
import numpy as np
import pandas as pd

from tqdm import tqdm

# =========================================================
# Gabor features + RBF kernel SVM
# =========================================================

DATA_DIR = "data/"

# -------------------------
# Load + reshape
# -------------------------
Xtr_df = pd.read_csv(DATA_DIR + "Xtr.csv", header=None)
Xte_df = pd.read_csv(DATA_DIR + "Xte.csv", header=None)
if Xtr_df.shape[1] == 3073:
    Xtr_df = Xtr_df.iloc[:, 1:]
if Xte_df.shape[1] == 3073:
    Xte_df = Xte_df.iloc[:, 1:]

Xtr_raw = Xtr_df.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)
Xte_raw = Xte_df.iloc[:, :3072].to_numpy(dtype=np.float32, copy=False)
Xtr_raw = np.nan_to_num(Xtr_raw, nan=0.0, posinf=0.0, neginf=0.0)
Xte_raw = np.nan_to_num(Xte_raw, nan=0.0, posinf=0.0, neginf=0.0)

Ytr_df = pd.read_csv(DATA_DIR + "Ytr.csv")
y = Ytr_df["Prediction"].to_numpy()

def to_rgb01(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32).reshape(-1, 3, 32, 32)
    if float(np.nanmax(X)) > 1.5:
        X = X / 255.0
    X = np.clip(X, 0.0, 1.0)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

Xtr_img = to_rgb01(Xtr_raw)
Xte_img = to_rgb01(Xte_raw)

# -------------------------
# Gabor feature extractor
# -------------------------
# Small, fast bank (expand if you want more capacity)
thetas = [0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
frequencies = [0.15, 0.30]  # cycles/pixel
bank = [(f, th) for f in frequencies for th in thetas]

def gabor_features(X: np.ndarray) -> np.ndarray:
    feats = np.empty((X.shape[0], 2 * len(bank)), dtype=np.float32)
    for i, img in enumerate(tqdm(X, desc="Gabor")):
        gray = img.mean(axis=0).astype(np.float32, copy=False)
        gray = np.nan_to_num(gray, nan=0.0, posinf=0.0, neginf=0.0)
        gray = np.clip(gray, 0.0, 1.0)
        k = 0
        for freq, theta in bank:
            real, imag = gabor(gray, frequency=freq, theta=theta)
            mag = np.sqrt(real * real + imag * imag).astype(np.float32, copy=False)
            feats[i, k] = float(np.mean(mag))
            feats[i, k + 1] = float(np.std(mag))
            k += 2
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

print("Extracting Gabor features...")
Xtr_gb = gabor_features(Xtr_img)
Xte_gb = gabor_features(Xte_img)

scaler = StandardScaler()
Xtr_gb = scaler.fit_transform(Xtr_gb)
Xte_gb = scaler.transform(Xte_gb)

# -------------------------
# Tune RBF SVM
# -------------------------
Cs = [0.3, 1.0, 3.0, 10.0]
gammas = ["scale", 0.1, 0.3, 1.0]
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

best = {"acc": -1.0, "C": None, "gamma": None}
for C in Cs:
    for gamma in gammas:
        fold_accs = []
        for tr_idx, va_idx in skf.split(Xtr_gb, y):
            clf = SVC(C=float(C), kernel="rbf", gamma=gamma)
            clf.fit(Xtr_gb[tr_idx], y[tr_idx])
            pred = clf.predict(Xtr_gb[va_idx])
            fold_accs.append(accuracy_score(y[va_idx], pred))
        mean_acc = float(np.mean(fold_accs))
        if mean_acc > best["acc"]:
            best = {"acc": mean_acc, "C": float(C), "gamma": gamma}
        print(f"C={C} gamma={gamma} | cv_acc={mean_acc:.4f} | best={best}")

print("Training final SVM...")
final_clf = SVC(C=float(best["C"]), kernel="rbf", gamma=best["gamma"])
final_clf.fit(Xtr_gb, y)
y_pred_test = final_clf.predict(Xte_gb)

out_path = "results/submission_gabor_rbf_svm.csv"
submission = pd.DataFrame({
    "Id": np.arange(1, len(y_pred_test) + 1),
    "Prediction": y_pred_test,
})
submission.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

## Fusion (Scattering RBF) + (HOPG RBF–Chi²)
On combine deux modèles forts :
- **Scattering** (grayscale) + **SVM RBF**
- **HOPG** (histogrammes) + **SVM RBF–Chi²**

> Deux fusions classiques :
>- **Kernel-sum ("concat" en espace de features)** : $K = \alpha K_{scat} + (1-\alpha)K_{hopg}$, puis un seul SVM sur $K$.
>- **Late fusion (moyenne des scores)** : on entraîne 2 SVM séparés et on moyenne leurs probabilités.

In [ ]:
# Fusion: (Scattering RBF) + (HOPG RBF–Chi²)
import numpy as np
import pandas as pd

from src.data import load_data, encode_labels

# ---------- Utils ----------
def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)

def sq_dists(X: np.ndarray, Z: np.ndarray) -> np.ndarray:
    """Squared Euclidean distances (n,m) in float32."""
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    Xn = np.sum(X * X, axis=1, keepdims=True)
    Zn = np.sum(Z * Z, axis=1, keepdims=True).T
    return (Xn + Zn - 2.0 * (X @ Z.T)).astype(np.float32, copy=False)

def estimate_gamma_rbf_from_sqdist(D2: np.ndarray) -> float:
    tri = D2[np.triu_indices_from(D2, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (2.0 * (med + 1e-12)))

def chi2_dist_block(Xb: np.ndarray, Zb: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X3 = Xb[:, None, :]
    Z3 = Zb[None, :, :]
    num = (X3 - Z3) ** 2
    den = X3 + Z3 + eps
    return np.sum(num / den, axis=2, dtype=np.float32)

def chi2_dist_matrix(
    X: np.ndarray,
    Z: np.ndarray,
    *,
    eps: float = 1e-12,
    chunk_x: int = 32,
    chunk_z: int = 256,
 ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n, m = X.shape[0], Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_x):
        i1 = min(i0 + chunk_x, n)
        Xb = X[i0:i1]
        for j0 in range(0, m, chunk_z):
            j1 = min(j0 + chunk_z, m)
            Zb = Z[j0:j1]
            D[i0:i1, j0:j1] = chi2_dist_block(Xb, Zb, eps=eps)
    return D

def estimate_gamma_chi2(X: np.ndarray, sample: int = 120, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D = np.empty((s, s), dtype=np.float32)
    for i in range(s):
        D[i] = chi2_dist_block(Xs[i : i + 1], Xs)[0]
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))

# ---------- Scattering features (reuse if available) ----------
def to_gray_images(X_flat: np.ndarray) -> np.ndarray:
    X = np.asarray(X_flat, dtype=np.float32).reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    if float(np.max(Xg)) > 1.5:
        Xg = Xg / 255.0
    return np.clip(Xg, 0.0, 1.0).astype(np.float32, copy=False)

def scattering_features_gray_fallback(Xg: np.ndarray, *, J: int = 2, L: int = 8, batch_size: int = 256) -> np.ndarray:
    from kymatio.scattering2d.frontend.numpy_frontend import ScatteringNumPy2D as Scattering2D
    scat = Scattering2D(J=J, shape=(32, 32), L=L)
    n = Xg.shape[0]
    out = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(Xg[i0:i1])
        out.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    return np.vstack(out)

# ---------- HOPG features (small, faster variant) ----------
def _phase_grad_best_response(
    img: np.ndarray,
    *,
    gabor_orientations: int = 4,
    frequencies: tuple[float, ...] = (0.25,),
    eps: float = 1e-12,
 ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    img = np.asarray(img, dtype=np.float32)
    H, W = img.shape
    thetas = np.linspace(0.0, np.pi, int(gabor_orientations), endpoint=False)
    best_mag = np.full((H, W), -np.inf, dtype=np.float32)
    best_gx = np.zeros((H, W), dtype=np.float32)
    best_gy = np.zeros((H, W), dtype=np.float32)
    for theta in thetas:
        for freq in frequencies:
            real, imag = gabor(img, frequency=float(freq), theta=float(theta))
            phase = np.arctan2(imag, real).astype(np.float32, copy=False)
            phase_u = np.unwrap(np.unwrap(phase, axis=0), axis=1).astype(np.float32, copy=False)
            gy, gx = np.gradient(phase_u)
            gx = gx.astype(np.float32, copy=False)
            gy = gy.astype(np.float32, copy=False)
            mag = np.sqrt(gx * gx + gy * gy + eps).astype(np.float32, copy=False)
            mask = mag > best_mag
            if np.any(mask):
                best_mag[mask] = mag[mask]
                best_gx[mask] = gx[mask]
                best_gy[mask] = gy[mask]
    best_mag = np.maximum(best_mag, 0.0).astype(np.float32, copy=False)
    return best_gx, best_gy, best_mag

def compute_hopg_features_fast(
    X_flat: np.ndarray,
    *,
    orientations: int = 9,
    pixels_per_cell: tuple[int, int] = (4, 4),
    cells_per_block: tuple[int, int] = (2, 2),
    gabor_orientations: int = 4,
    frequencies: tuple[float, ...] = (0.25,),
 ) -> np.ndarray:
    Xg = to_gray_images(X_flat)
    n, H, W = Xg.shape
    cy, cx = pixels_per_cell
    by, bx = cells_per_block
    n_cells_y = H // cy
    n_cells_x = W // cx
    n_bins = int(orientations)
    bin_edges = np.linspace(0.0, 2.0 * np.pi, n_bins + 1, endpoint=True).astype(np.float32)
    cell_hist = np.zeros((n, n_cells_y, n_cells_x, n_bins), dtype=np.float32)
    for i in range(n):
        gx, gy, mag = _phase_grad_best_response(
            Xg[i], gabor_orientations=gabor_orientations, frequencies=frequencies
        )
        ang = (np.arctan2(gy, gx) % (2.0 * np.pi)).astype(np.float32, copy=False)
        for yy in range(n_cells_y):
            y0, y1 = yy * cy, (yy + 1) * cy
            for xx in range(n_cells_x):
                x0, x1 = xx * cx, (xx + 1) * cx
                a = ang[y0:y1, x0:x1].ravel()
                w = mag[y0:y1, x0:x1].ravel()
                b = np.clip(np.searchsorted(bin_edges, a, side="right") - 1, 0, n_bins - 1)
                cell_hist[i, yy, xx] = np.bincount(b, weights=w, minlength=n_bins).astype(np.float32, copy=False)
    # block L2 normalization
    eps = 1e-6
    n_blocks_y = n_cells_y - by + 1
    n_blocks_x = n_cells_x - bx + 1
    blocks = []
    for yy in range(n_blocks_y):
        for xx in range(n_blocks_x):
            block = cell_hist[:, yy : yy + by, xx : xx + bx, :].reshape(n, -1)
            norm = np.sqrt(np.sum(block * block, axis=1, keepdims=True) + eps)
            blocks.append(block / norm)
    feats = np.concatenate(blocks, axis=1).astype(np.float32, copy=False)
    feats = np.maximum(feats, 0.0)
    return feats

# ---------- Run fusion pipeline ----------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

# Keep it modest for ~5 minutes
subset_size = 800
test_size = 0.2
seed = 42
X_sub_raw = X_raw[:subset_size]
y_sub = y_int[:subset_size]

# 1) Scattering features (standardized)
if "X_sub_scat_s" in globals() and "X_test_scat_s" in globals() and isinstance(globals()["X_sub_scat_s"], np.ndarray):
    # Reuse from the existing Scattering+RBF cell if available
    Xs_sub = globals()["X_sub_scat_s"][:subset_size].astype(np.float32, copy=False)
    Xs_test = globals()["X_test_scat_s"].astype(np.float32, copy=False)
    print("Reusing precomputed scattering features from previous cell.")
else:
    print("Computing scattering features (fallback)...")
    Xg_sub = to_gray_images(X_sub_raw)
    Xg_test = to_gray_images(X_test_raw)
    Xs_sub_raw = scattering_features_gray_fallback(Xg_sub, J=2, L=8, batch_size=256)
    Xs_test_raw = scattering_features_gray_fallback(Xg_test, J=2, L=8, batch_size=256)
    scaler_s = StandardScaler(with_mean=True, with_std=True)
    Xs_sub = scaler_s.fit_transform(Xs_sub_raw).astype(np.float32, copy=False)
    Xs_test = scaler_s.transform(Xs_test_raw).astype(np.float32, copy=False)

# 2) HOPG features (Chi² kernel)
print("Computing HOPG features...")
Xh_sub = compute_hopg_features_fast(X_sub_raw, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2))
Xh_test = compute_hopg_features_fast(X_test_raw, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2))
Xh_sub = l1_normalize_nonneg(Xh_sub)
Xh_test = l1_normalize_nonneg(Xh_test)

# 3) Shared train/val split indices (IMPORTANT: align both modalities)
rng = np.random.default_rng(seed)
perm = rng.permutation(subset_size)
n_val = int(round(test_size * subset_size))
val_idx = perm[:n_val]
tr_idx = perm[n_val:]

Xs_tr, Xs_va, ys_tr, ys_va = Xs_sub[tr_idx], Xs_sub[val_idx], y_sub[tr_idx], y_sub[val_idx]
Xh_tr, Xh_va, yh_tr, yh_va = Xh_sub[tr_idx], Xh_sub[val_idx], y_sub[tr_idx], y_sub[val_idx]

# 4) Precompute distances/kernels
print("Precomputing Scattering RBF distances...")
D2_s_tr = sq_dists(Xs_tr, Xs_tr)
D2_s_va = sq_dists(Xs_va, Xs_tr)

gamma_s = estimate_gamma_rbf_from_sqdist(D2_s_tr)
print(f"gamma_s (RBF) = {gamma_s:.3e}")
K_s_tr = np.exp(-gamma_s * D2_s_tr).astype(np.float32, copy=False)
K_s_va = np.exp(-gamma_s * D2_s_va).astype(np.float32, copy=False)

print("Precomputing HOPG Chi² distances...")
D_h_tr = chi2_dist_matrix(Xh_tr, Xh_tr, chunk_x=32, chunk_z=256)
D_h_va = chi2_dist_matrix(Xh_va, Xh_tr, chunk_x=32, chunk_z=256)
gamma_h = estimate_gamma_chi2(Xh_tr, sample=min(120, Xh_tr.shape[0]), seed=0)
print(f"gamma_h (Chi²) = {gamma_h:.3e}")
K_h_tr = np.exp(-gamma_h * D_h_tr).astype(np.float32, copy=False)
K_h_va = np.exp(-gamma_h * D_h_va).astype(np.float32, copy=False)

# 5) Tune C for each single model (tiny grid)
Cs = [0.3, 1.0, 3.0, 10.0]
best_s = {"acc": -1.0}
best_h = {"acc": -1.0}
for C in Cs:
    clf_s = SVC(C=float(C), kernel="precomputed")
    clf_s.fit(K_s_tr, ys_tr)
    acc_s = float(np.mean(clf_s.predict(K_s_va) == ys_va))
    print(f"[Scat] C={C:.3g} val_acc={acc_s:.4f}")
    if acc_s > best_s["acc"]:
        best_s = {"acc": acc_s, "C": float(C)}
    clf_h = SVC(C=float(C), kernel="precomputed")
    clf_h.fit(K_h_tr, yh_tr)
    acc_h = float(np.mean(clf_h.predict(K_h_va) == yh_va))
    print(f"[HOPG] C={C:.3g} val_acc={acc_h:.4f}")
    if acc_h > best_h["acc"]:
        best_h = {"acc": acc_h, "C": float(C)}

print("Best single-model params:", {"scat": best_s, "hopg": best_h})

# 6) Kernel-sum fusion (kernel-only 'concat')
alphas = [0.25, 0.5, 0.75]
best_f = {"acc": -1.0}
for a in alphas:
    Kf_tr = (float(a) * K_s_tr + (1.0 - float(a)) * K_h_tr).astype(np.float32, copy=False)
    Kf_va = (float(a) * K_s_va + (1.0 - float(a)) * K_h_va).astype(np.float32, copy=False)
    for C in Cs:
        clf_f = SVC(C=float(C), kernel="precomputed")
        clf_f.fit(Kf_tr, ys_tr)
        acc_f = float(np.mean(clf_f.predict(Kf_va) == ys_va))
        print(f"[Kernel-sum] a={a:.2f} C={C:.3g} val_acc={acc_f:.4f}")
        if acc_f > best_f["acc"]:
            best_f = {"acc": acc_f, "alpha": float(a), "C": float(C)}

print("Best fusion (kernel-sum):", best_f)

# 7) Fit final models on full subset and predict test
print("Building full kernels (subset/test)...")
D2_s_sub = sq_dists(Xs_sub, Xs_sub)
D2_s_te = sq_dists(Xs_test, Xs_sub)
K_s_sub = np.exp(-gamma_s * D2_s_sub).astype(np.float32, copy=False)
K_s_te = np.exp(-gamma_s * D2_s_te).astype(np.float32, copy=False)

D_h_sub = chi2_dist_matrix(Xh_sub, Xh_sub, chunk_x=32, chunk_z=256)
D_h_te = chi2_dist_matrix(Xh_test, Xh_sub, chunk_x=32, chunk_z=256)
K_h_sub = np.exp(-gamma_h * D_h_sub).astype(np.float32, copy=False)
K_h_te = np.exp(-gamma_h * D_h_te).astype(np.float32, copy=False)

# (A) Kernel-sum single SVM
Kf_sub = (best_f["alpha"] * K_s_sub + (1.0 - best_f["alpha"]) * K_h_sub).astype(np.float32, copy=False)
Kf_te = (best_f["alpha"] * K_s_te + (1.0 - best_f["alpha"]) * K_h_te).astype(np.float32, copy=False)
clf_f_final = SVC(C=best_f["C"], kernel="precomputed")
clf_f_final.fit(Kf_sub, y_sub)
y_f_int = clf_f_final.predict(Kf_te)
y_f = np.array([inv_map[int(i)] for i in y_f_int])
sub_f = pd.DataFrame({"Prediction": y_f})
sub_f.index += 1
out_path_f = "results/submission_fusion_kernel_sum_scattering_hopg.csv"
sub_f.to_csv(out_path_f, index_label="Id")
print(f"Saved kernel-sum fusion to {out_path_f}")

# (B) Late fusion: average probabilities from two SVMs
# Note: probability=True makes fitting slower, so we only do it once (after selecting C).
print("Training probabilistic SVMs for late fusion...")
clf_s_prob = SVC(C=best_s["C"], kernel="precomputed", probability=True)
clf_h_prob = SVC(C=best_h["C"], kernel="precomputed", probability=True)
clf_s_prob.fit(K_s_sub, y_sub)
clf_h_prob.fit(K_h_sub, y_sub)
P_s = clf_s_prob.predict_proba(K_s_te)
P_h = clf_h_prob.predict_proba(K_h_te)
P = 0.5 * P_s + 0.5 * P_h
classes = clf_s_prob.classes_
y_late_int = classes[np.argmax(P, axis=1)]
y_late = np.array([inv_map[int(i)] for i in y_late_int])
sub_l = pd.DataFrame({"Prediction": y_late})
sub_l.index += 1
out_path_l = "results/submission_fusion_avgproba_scattering_hopg.csv"
sub_l.to_csv(out_path_l, index_label="Id")
print(f"Saved late-fusion (avg proba) to {out_path_l}")
print(f"subset_size={subset_size} alpha={best_f['alpha']:.2f}")

## Fusion (1 seul SVM) : HOG (RBF) + Scattering (RBF–Chi²)

Ici on entraîne **un seul SVM** sur un **noyau combiné** (kernel-sum) :
- $K_{hog}(x,z)=\exp(-\gamma_{hog}\,\|h(x)-h(z)\|^2)$ avec features HOG.
- $K_{scat}(x,z)=\exp(-\gamma_{scat}\,\chi^2(s(x),s(z)))$ avec features Scattering2D (non-négatives, normalisées $\ell_1$).
- Fusion : $K=\alpha K_{hog} + (1-\alpha) K_{scat}$, puis `SVC(kernel="precomputed")`.

Notes pratiques :
- HOG est calculé avec une config “riche”, puis **mis à taille 3500** en prenant les 3500 premières dimensions (ou padding si besoin).
- Le tuning est hiérarchique : on tune $(\gamma,C)$ séparément pour HOG et Scattering, puis on tune $(\alpha,C)$ pour la fusion avec ces $\gamma$ fixés.

In [ ]:
# 1-SVM fusion: HOG (RBF) + Scattering (RBF–Chi²) — tuned + HOG dim=3500
import numpy as np
import pandas as pd

from src.data import load_data, encode_labels
from src.features import compute_hog_features

# ---------- Helpers ----------
def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)

def to_gray_images(X_flat: np.ndarray) -> np.ndarray:
    X = np.asarray(X_flat, dtype=np.float32).reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    if float(np.max(Xg)) > 1.5:
        Xg = Xg / 255.0
    return np.clip(Xg, 0.0, 1.0).astype(np.float32, copy=False)

def scattering_features_gray(Xg: np.ndarray, *, J: int = 2, L: int = 8, batch_size: int = 256) -> np.ndarray:
    from kymatio.scattering2d.frontend.numpy_frontend import ScatteringNumPy2D as Scattering2D
    scat = Scattering2D(J=J, shape=(32, 32), L=L)
    n = Xg.shape[0]
    out = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(Xg[i0:i1])
        out.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    return np.vstack(out)

def sq_dists(X: np.ndarray, Z: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    Xn = np.sum(X * X, axis=1, keepdims=True)
    Zn = np.sum(Z * Z, axis=1, keepdims=True).T
    return (Xn + Zn - 2.0 * (X @ Z.T)).astype(np.float32, copy=False)

def estimate_gamma_rbf_from_sqdist(D2: np.ndarray) -> float:
    tri = D2[np.triu_indices_from(D2, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (2.0 * (med + 1e-12)))

def chi2_dist_block(Xb: np.ndarray, Zb: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X3 = Xb[:, None, :]
    Z3 = Zb[None, :, :]
    num = (X3 - Z3) ** 2
    den = X3 + Z3 + eps
    return np.sum(num / den, axis=2, dtype=np.float32)

def chi2_dist_matrix(
    X: np.ndarray,
    Z: np.ndarray,
    *,
    eps: float = 1e-12,
    chunk_x: int = 32,
    chunk_z: int = 256,
 ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    n, m = X.shape[0], Z.shape[0]
    D = np.empty((n, m), dtype=np.float32)
    for i0 in range(0, n, chunk_x):
        i1 = min(i0 + chunk_x, n)
        Xb = X[i0:i1]
        for j0 in range(0, m, chunk_z):
            j1 = min(j0 + chunk_z, m)
            Zb = Z[j0:j1]
            D[i0:i1, j0:j1] = chi2_dist_block(Xb, Zb, eps=eps)
    return D

def estimate_gamma_chi2(X: np.ndarray, sample: int = 120, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D = np.empty((s, s), dtype=np.float32)
    for i in range(s):
        D[i] = chi2_dist_block(Xs[i : i + 1], Xs)[0]
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))

def force_dim(X: np.ndarray, d: int) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    n, p = X.shape
    if p == d:
        return X
    if p > d:
        return X[:, :d].astype(np.float32, copy=False)
    pad = d - p
    return np.pad(X, ((0, 0), (0, pad)), mode="constant").astype(np.float32, copy=False)

# ---------- Data ----------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

subset_size = 800
test_size = 0.2
seed = 42
X_sub_raw = X_raw[:subset_size]
y_sub = y_int[:subset_size]

# ---------- Features ----------
target_hog_dim = 3500

print("Computing HOG (rich)...")
# This HOG config yields ~3600 dims on 32x32 images; we force it to exactly 3500 afterwards.
hog_params = dict(orientations=9, pixels_per_cell=(4, 4), cells_per_block=(4, 4))
Xhog_sub = compute_hog_features(X_sub_raw, **hog_params)
Xhog_test = compute_hog_features(X_test_raw, **hog_params)
print(f"Raw HOG dim = {Xhog_sub.shape[1]}")

scaler_h = StandardScaler(with_mean=True, with_std=True)
Xhog_sub_s = scaler_h.fit_transform(Xhog_sub).astype(np.float32, copy=False)
Xhog_test_s = scaler_h.transform(Xhog_test).astype(np.float32, copy=False)

Xhog_sub_s = force_dim(Xhog_sub_s, target_hog_dim)
Xhog_test_s = force_dim(Xhog_test_s, target_hog_dim)
print(f"Final HOG dim = {Xhog_sub_s.shape[1]}")

print("Computing Scattering...")
if "X_sub_scat" in globals() and "X_test_scat" in globals() and isinstance(globals()["X_sub_scat"], np.ndarray):
    Xscat_sub_raw = globals()["X_sub_scat"][:subset_size].astype(np.float32, copy=False)
    Xscat_test_raw = globals()["X_test_scat"].astype(np.float32, copy=False)
    print("Reusing precomputed scattering (raw, non-standardized).")
else:
    Xg_sub = to_gray_images(X_sub_raw)
    Xg_test = to_gray_images(X_test_raw)
    Xscat_sub_raw = scattering_features_gray(Xg_sub, J=2, L=8, batch_size=256)
    Xscat_test_raw = scattering_features_gray(Xg_test, J=2, L=8, batch_size=256)
Xscat_sub = l1_normalize_nonneg(Xscat_sub_raw)
Xscat_test = l1_normalize_nonneg(Xscat_test_raw)
print(f"Scattering dim = {Xscat_sub.shape[1]}")

# ---------- Shared split (align modalities) ----------
rng = np.random.default_rng(seed)
perm = rng.permutation(subset_size)
n_val = int(round(test_size * subset_size))
val_idx = perm[:n_val]
tr_idx = perm[n_val:]

Xhog_tr, Xhog_va, y_tr, y_va = Xhog_sub_s[tr_idx], Xhog_sub_s[val_idx], y_sub[tr_idx], y_sub[val_idx]
Xsc_tr, Xsc_va = Xscat_sub[tr_idx], Xscat_sub[val_idx]

# ---------- Distances (compute once), kernels vary by gamma ----------
print("Precomputing distances (train/val)...")
D2_h_tr = sq_dists(Xhog_tr, Xhog_tr)
D2_h_va = sq_dists(Xhog_va, Xhog_tr)
D_sc_tr = chi2_dist_matrix(Xsc_tr, Xsc_tr, chunk_x=32, chunk_z=256)
D_sc_va = chi2_dist_matrix(Xsc_va, Xsc_tr, chunk_x=32, chunk_z=256)

gamma_hog0 = estimate_gamma_rbf_from_sqdist(D2_h_tr)
gamma_scat0 = estimate_gamma_chi2(Xsc_tr, sample=min(160, Xsc_tr.shape[0]), seed=0)
print(f"gamma_hog0={gamma_hog0:.3e}  gamma_scat0={gamma_scat0:.3e}")

# ---------- Tuning grids ----------
Cs = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
gamma_factors = [0.25, 0.5, 1.0, 2.0, 4.0]
alphas = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]

# ---------- 1) Tune HOG (gamma, C) ----------
best_hog = {"acc": -1.0}
for gf in gamma_factors:
    g = float(gamma_hog0 * gf)
    K_tr = np.exp(-g * D2_h_tr).astype(np.float32, copy=False)
    K_va = np.exp(-g * D2_h_va).astype(np.float32, copy=False)
    for C in Cs:
        clf = SVC(C=float(C), kernel="precomputed")
        clf.fit(K_tr, y_tr)
        acc = float(np.mean(clf.predict(K_va) == y_va))
        if acc > best_hog["acc"]:
            best_hog = {"acc": acc, "gamma": g, "C": float(C)}
print("Best HOG:", best_hog)

# ---------- 2) Tune Scattering (gamma, C) ----------
best_scat = {"acc": -1.0}
for gf in gamma_factors:
    g = float(gamma_scat0 * gf)
    K_tr = np.exp(-g * D_sc_tr).astype(np.float32, copy=False)
    K_va = np.exp(-g * D_sc_va).astype(np.float32, copy=False)
    for C in Cs:
        clf = SVC(C=float(C), kernel="precomputed")
        clf.fit(K_tr, y_tr)
        acc = float(np.mean(clf.predict(K_va) == y_va))
        if acc > best_scat["acc"]:
            best_scat = {"acc": acc, "gamma": g, "C": float(C)}
print("Best Scattering:", best_scat)

# ---------- 3) Tune fusion (alpha, C) with gammas fixed ----------
K_h_tr = np.exp(-best_hog["gamma"] * D2_h_tr).astype(np.float32, copy=False)
K_h_va = np.exp(-best_hog["gamma"] * D2_h_va).astype(np.float32, copy=False)
K_s_tr = np.exp(-best_scat["gamma"] * D_sc_tr).astype(np.float32, copy=False)
K_s_va = np.exp(-best_scat["gamma"] * D_sc_va).astype(np.float32, copy=False)

best = {"acc": -1.0}
for a in alphas:
    Ktr = (float(a) * K_h_tr + (1.0 - float(a)) * K_s_tr).astype(np.float32, copy=False)
    Kva = (float(a) * K_h_va + (1.0 - float(a)) * K_s_va).astype(np.float32, copy=False)
    for C in Cs:
        clf = SVC(C=float(C), kernel="precomputed")
        clf.fit(Ktr, y_tr)
        acc = float(np.mean(clf.predict(Kva) == y_va))
        if acc > best["acc"]:
            best = {"acc": acc, "alpha": float(a), "C": float(C)}
print("Best Fusion:", best)

# ---------- Final train on full subset & predict test ----------
print("Building full kernels (subset/test)...")
D2_h_sub = sq_dists(Xhog_sub_s, Xhog_sub_s)
D2_h_te = sq_dists(Xhog_test_s, Xhog_sub_s)
K_h_sub = np.exp(-best_hog["gamma"] * D2_h_sub).astype(np.float32, copy=False)
K_h_te = np.exp(-best_hog["gamma"] * D2_h_te).astype(np.float32, copy=False)

D_sc_sub = chi2_dist_matrix(Xscat_sub, Xscat_sub, chunk_x=32, chunk_z=256)
D_sc_te = chi2_dist_matrix(Xscat_test, Xscat_sub, chunk_x=32, chunk_z=256)
K_s_sub = np.exp(-best_scat["gamma"] * D_sc_sub).astype(np.float32, copy=False)
K_s_te = np.exp(-best_scat["gamma"] * D_sc_te).astype(np.float32, copy=False)

K_sub = (best["alpha"] * K_h_sub + (1.0 - best["alpha"]) * K_s_sub).astype(np.float32, copy=False)
K_te = (best["alpha"] * K_h_te + (1.0 - best["alpha"]) * K_s_te).astype(np.float32, copy=False)

final_clf = SVC(C=best["C"], kernel="precomputed")
final_clf.fit(K_sub, y_sub)
yte_int = final_clf.predict(K_te)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_single_svm_hog3500_rbf_plus_scattering_rbfchi2_tuned.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print({"subset_size": subset_size, "hog_dim": int(Xhog_sub_s.shape[1]), **best_hog, **{f"scat_{k}": v for k, v in best_scat.items()}, **best})

## SIFT + Scattering + SVM (fusion de noyaux)

On combine deux familles de features :
- **SIFT** (local) → représentation fixe via **Bag-of-Visual-Words** (histogramme de “visual words”)
- **Scattering transform** (global) → représentation stable aux petites déformations

On entraîne un **SVM à noyau pré-calculé** avec une **fusion** :
- $K_{\text{sift}}(x,z)=\exp(-\gamma_{\text{sift}}\,\chi^2(x,z))$ (RBF–Chi²)
- $K_{\text{scat}}(x,z)=\exp(-\gamma_{\text{scat}}\,\|x-z\|_2^2)$ (RBF)
- $K = \alpha K_{\text{sift}} + (1-\alpha) K_{\text{scat}}$

Tuning demandé : **grid search réduit** sur un petit subset, puis **entraînement final plus long** (subset plus grand) avec les hyperparamètres trouvés.

In [ ]:
# SIFT (BoVW) + Scattering2D + SVM (precomputed kernel fusion)
# Strategy: reduced grid search on a small subset, then train longer on a larger subset.

import numpy as np
import pandas as pd

from src.data import load_data, encode_labels


# -----------------------------
# Utilities
# -----------------------------

def to_gray_images(X_flat: np.ndarray) -> np.ndarray:
    """(n, 3072) -> (n, 32, 32) float32 in [0,1]."""
    X = np.asarray(X_flat, dtype=np.float32)
    X = X.reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    mx = float(np.max(Xg))
    if mx > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0)
    return Xg.astype(np.float32, copy=False)


def stratified_split_idx(y: np.ndarray, test_size: float, seed: int):
    y = np.asarray(y)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=float(test_size), random_state=int(seed))
    tr_idx, va_idx = next(sss.split(np.zeros_like(y), y))
    return tr_idx, va_idx


def scattering_features(X_img: np.ndarray, *, J: int = 2, L: int = 8, batch_size: int = 256) -> np.ndarray:
    scat = Scattering2D(J=J, shape=(32, 32), L=L)
    n = X_img.shape[0]
    feats = []
    for i0 in range(0, n, batch_size):
        i1 = min(i0 + batch_size, n)
        S = scat(X_img[i0:i1])
        feats.append(S.reshape(S.shape[0], -1).astype(np.float32, copy=False))
    return np.vstack(feats)


def l1_normalize_nonneg(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0)
    return (X / (np.sum(X, axis=1, keepdims=True) + eps)).astype(np.float32, copy=False)


def extract_sift_descriptors(img_gray_01: np.ndarray, *, max_desc: int, rng: np.random.Generator) -> np.ndarray:
    """Return (n_desc, 128) float32. Image expected float in [0,1]."""
    img = np.asarray(img_gray_01, dtype=np.float32)
    try:
        # Lower c_dog => more keypoints on low-contrast images
        sift = SIFT(c_dog=0.008)
        sift.detect_and_extract(img)
        desc = sift.descriptors
    except Exception:
        desc = None

    if desc is None or len(desc) == 0:
        return np.empty((0, 128), dtype=np.float32)

    desc = np.asarray(desc, dtype=np.float32)
    if max_desc is not None and desc.shape[0] > int(max_desc):
        idx = rng.choice(desc.shape[0], size=int(max_desc), replace=False)
        desc = desc[idx]
    return desc


def fit_sift_codebook(
    X_gray: np.ndarray,
    *,
    n_clusters: int,
    max_desc_per_image: int,
    max_images: int,
    seed: int,
    target_desc: int | None = None,
    min_clusters: int = 16,
 ) -> MiniBatchKMeans:
    """Fit a BoVW codebook robustly, even if few descriptors are found.
    """
    rng = np.random.default_rng(seed)
    n = int(X_gray.shape[0])
    m = int(min(max_images, n))
    perm = rng.permutation(n)[:m]

    if target_desc is None:
        # Aim for a reasonable number of descriptors without being too slow
        target_desc = int(max(2000, 50 * int(n_clusters)))

    descs = []
    total = 0
    for i in perm:
        d = extract_sift_descriptors(X_gray[i], max_desc=max_desc_per_image, rng=rng)
        if d.shape[0] == 0:
            continue
        descs.append(d)
        total += int(d.shape[0])
        if total >= int(target_desc):
            break

    if len(descs) == 0:
        raise RuntimeError("No SIFT descriptors found on the sampled images.")

    D = np.vstack(descs)
    n_desc = int(D.shape[0])
    n_clusters_eff = int(min(int(n_clusters), n_desc))
    n_clusters_eff = int(max(int(min_clusters), n_clusters_eff))
    n_clusters_eff = int(min(n_clusters_eff, n_desc))
    if n_clusters_eff < int(min_clusters):
        raise RuntimeError(
            f"Too few SIFT descriptors for BoVW: n_desc={n_desc}. "
            f"Try increasing subset_size_tune/max_desc_per_image or lowering n_clusters."
        )

    if n_clusters_eff != int(n_clusters):
        print(f"[BoVW] Reducing n_clusters: {n_clusters} -> {n_clusters_eff} (n_desc={n_desc})")
    else:
        print(f"[BoVW] Using n_clusters={n_clusters_eff} (n_desc={n_desc})")

    kmeans = MiniBatchKMeans(
        n_clusters=int(n_clusters_eff),
        batch_size=4096,
        random_state=int(seed),
        n_init="auto",
        reassignment_ratio=0.01,
    )
    kmeans.fit(D)
    return kmeans


def sift_bovw_hist(
    X_gray: np.ndarray,
    kmeans: MiniBatchKMeans,
    *,
    max_desc_per_image: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = X_gray.shape[0]
    k = int(kmeans.n_clusters)

    H = np.zeros((n, k), dtype=np.float32)
    for i in range(n):
        desc = extract_sift_descriptors(X_gray[i], max_desc=max_desc_per_image, rng=rng)
        if desc.shape[0] == 0:
            continue
        words = kmeans.predict(desc)
        H[i] = np.bincount(words, minlength=k).astype(np.float32)

    return l1_normalize_nonneg(H)


def estimate_gamma_rbf_sqeuclidean(X: np.ndarray, sample: int = 400, seed: int = 0) -> float:
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, X.shape[0]))
    idx = rng.choice(X.shape[0], size=s, replace=False)
    Xs = X[idx]
    D2 = pairwise_distances(Xs, Xs, metric="sqeuclidean")
    tri = D2[np.triu_indices_from(D2, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))


def estimate_gamma_chi2_from_hist(H: np.ndarray, sample: int = 400, seed: int = 0) -> float:
    """Median-heuristic gamma for exp(-gamma * chi2). Uses chi2_kernel(gamma=1) => K=exp(-D)."""
    H = np.asarray(H, dtype=np.float32)
    rng = np.random.default_rng(seed)
    s = int(min(sample, H.shape[0]))
    idx = rng.choice(H.shape[0], size=s, replace=False)
    Hs = H[idx]
    K = chi2_kernel(Hs, Hs, gamma=1.0)
    D = (-np.log(np.clip(K, 1e-12, 1.0))).astype(np.float32, copy=False)
    tri = D[np.triu_indices_from(D, k=1)]
    med = float(np.median(tri))
    return float(1.0 / (med + 1e-12))


# -----------------------------
# 1) Load + preprocess
# -----------------------------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

X_gray = to_gray_images(X_raw)
X_test_gray = to_gray_images(X_test_raw)

# -----------------------------
# 2) Reduced tuning grid on a small subset
# -----------------------------
seed = 42
val_size = 0.2
subset_size_tune = 1200

# SIFT-BoVW parameters (fixed here to keep tuning budget small)
# If you want to explore feature length, try n_clusters in {64, 128, 256}.
n_clusters = 128
max_desc_per_image = 80
max_images_for_vocab = subset_size_tune

X_tune_gray = X_gray[:subset_size_tune]
y_tune = y_int[:subset_size_tune]
tr_idx, va_idx = stratified_split_idx(y_tune, test_size=val_size, seed=seed)

print("Fitting SIFT codebook (BoVW)...")
kmeans = fit_sift_codebook(
    X_tune_gray,
    n_clusters=n_clusters,
    max_desc_per_image=max_desc_per_image,
    max_images=max_images_for_vocab,
    seed=seed,
    target_desc=max(5000, 80 * n_clusters),
    min_clusters=16,
 )
n_clusters_eff = int(kmeans.n_clusters)
print({"n_clusters_requested": int(n_clusters), "n_clusters_used": n_clusters_eff})

print("Computing SIFT BoVW histograms (train/val)...")
H_tune = sift_bovw_hist(X_tune_gray, kmeans, max_desc_per_image=max_desc_per_image, seed=seed)
H_tr = H_tune[tr_idx]
H_va = H_tune[va_idx]

print("Computing scattering features (train/val)...")
S_tune = scattering_features(X_tune_gray, J=2, L=8, batch_size=256)
S_tr_raw = S_tune[tr_idx]
S_va_raw = S_tune[va_idx]

# Standardize scattering features for the RBF kernel
scaler = StandardScaler(with_mean=True, with_std=True)
S_tr = scaler.fit_transform(S_tr_raw).astype(np.float32, copy=False)
S_va = scaler.transform(S_va_raw).astype(np.float32, copy=False)

y_tr = y_tune[tr_idx]
y_va = y_tune[va_idx]

# Precompute distances once for fast grid search
print("Precomputing distances for grid search...")
# For SIFT: get D_chi2 via chi2_kernel(gamma=1) => D = -log(K)
K1_tr = chi2_kernel(H_tr, H_tr, gamma=1.0)
D_sift_tr = (-np.log(np.clip(K1_tr, 1e-12, 1.0))).astype(np.float32, copy=False)
K1_va = chi2_kernel(H_va, H_tr, gamma=1.0)
D_sift_va = (-np.log(np.clip(K1_va, 1e-12, 1.0))).astype(np.float32, copy=False)

# For scattering: squared euclidean distances
D2_scat_tr = pairwise_distances(S_tr, S_tr, metric="sqeuclidean").astype(np.float32, copy=False)
D2_scat_va = pairwise_distances(S_va, S_tr, metric="sqeuclidean").astype(np.float32, copy=False)

# Median-heuristic gammas (base)
gamma_sift0 = estimate_gamma_chi2_from_hist(H_tr, sample=min(400, H_tr.shape[0]), seed=seed)
gamma_scat0 = estimate_gamma_rbf_sqeuclidean(S_tr, sample=min(400, S_tr.shape[0]), seed=seed)
print({"gamma_sift0": gamma_sift0, "gamma_scat0": gamma_scat0})

# Reduced grids
Cs = [1.0, 3.0, 10.0]
gf = [0.5, 1.0, 2.0]
alphas = [0.3, 0.5, 0.7]

best = {"acc": -1.0}

print("Running reduced grid search...")
for alpha in alphas:
    for C in Cs:
        for gfs in gf:
            for gfc in gf:
                gamma_sift = float(gamma_sift0 * gfs)
                gamma_scat = float(gamma_scat0 * gfc)

                K_sift_tr = np.exp(-gamma_sift * D_sift_tr).astype(np.float32, copy=False)
                K_sift_va = np.exp(-gamma_sift * D_sift_va).astype(np.float32, copy=False)

                K_scat_tr = np.exp(-gamma_scat * D2_scat_tr).astype(np.float32, copy=False)
                K_scat_va = np.exp(-gamma_scat * D2_scat_va).astype(np.float32, copy=False)

                K_tr = (alpha * K_sift_tr + (1.0 - alpha) * K_scat_tr).astype(np.float32, copy=False)
                K_va = (alpha * K_sift_va + (1.0 - alpha) * K_scat_va).astype(np.float32, copy=False)

                clf = SVC(C=float(C), kernel="precomputed", cache_size=2000)
                clf.fit(K_tr, y_tr)
                acc = float(np.mean(clf.predict(K_va) == y_va))

                if acc > best["acc"]:
                    best = {
                        "acc": acc,
                        "alpha": float(alpha),
                        "C": float(C),
                        "gamma_sift": float(gamma_sift),
                        "gamma_scat": float(gamma_scat),
                    }

print("Best params on tuning subset:", best)

# -----------------------------
# 3) Longer final training on a larger subset
# -----------------------------
subset_size_final = int(min(5000, X_gray.shape[0]))

X_sub_gray = X_gray[:subset_size_final]
y_sub = y_int[:subset_size_final]

print("\nComputing final SIFT BoVW histograms (subset + test)...")
H_sub = sift_bovw_hist(X_sub_gray, kmeans, max_desc_per_image=max_desc_per_image, seed=seed)
H_test = sift_bovw_hist(X_test_gray, kmeans, max_desc_per_image=max_desc_per_image, seed=seed)

print("Computing final scattering features (subset + test)...")
S_sub_raw = scattering_features(X_sub_gray, J=2, L=8, batch_size=256)
S_test_raw = scattering_features(X_test_gray, J=2, L=8, batch_size=256)

scaler_final = StandardScaler(with_mean=True, with_std=True)
S_sub = scaler_final.fit_transform(S_sub_raw).astype(np.float32, copy=False)
S_test = scaler_final.transform(S_test_raw).astype(np.float32, copy=False)

alpha = float(best["alpha"])
C = float(best["C"])
gamma_sift = float(best["gamma_sift"])
gamma_scat = float(best["gamma_scat"])

print("Precomputing fused Gram matrices (subset/subset + test/subset)...")
K_sift_sub = chi2_kernel(H_sub, H_sub, gamma=gamma_sift).astype(np.float32, copy=False)
K_sift_test = chi2_kernel(H_test, H_sub, gamma=gamma_sift).astype(np.float32, copy=False)

K_scat_sub = rbf_kernel(S_sub, S_sub, gamma=gamma_scat).astype(np.float32, copy=False)
K_scat_test = rbf_kernel(S_test, S_sub, gamma=gamma_scat).astype(np.float32, copy=False)

K_sub = (alpha * K_sift_sub + (1.0 - alpha) * K_scat_sub).astype(np.float32, copy=False)
K_test = (alpha * K_sift_test + (1.0 - alpha) * K_scat_test).astype(np.float32, copy=False)

final_clf = SVC(C=C, kernel="precomputed", cache_size=4000)
final_clf.fit(K_sub, y_sub)

yte_int = final_clf.predict(K_test)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_sift_scattering_fusion_svm.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print({"subset_size_tune": subset_size_tune, "subset_size_final": subset_size_final, "n_clusters_requested": int(n_clusters), "n_clusters_used": n_clusters_eff, "max_desc_per_image": int(max_desc_per_image), **best})

## Dense SIFT + Fisher Vector + Linear SVM

Pipeline :
- **Dense SIFT** : descripteurs SIFT calculés sur une **grille régulière** (pas `step`).
- **Fisher Vector** : agrégation de l’ensemble des descripteurs d’une image via un **GMM** à `K` composantes (vecteur fixe de dimension $2KD$).
- **Linear SVM** : classification sur les Fisher vectors.

Tuning : **grid réduit** sur un petit subset (quelques valeurs de `K`, `step`, et `C`), puis **entraînement final plus long** sur un subset plus grand avec les meilleurs hyperparamètres.

In [ ]:
# Dense SIFT + Fisher Vector + Linear SVM
# Reduced tuning grid, then longer final training.

import numpy as np
import pandas as pd

from src.data import load_data, encode_labels


# -----------------------------
# Helpers
# -----------------------------

def to_gray_u8_images(X_flat: np.ndarray) -> np.ndarray:
    """(n, 3072) -> (n, 32, 32) uint8."""
    X = np.asarray(X_flat, dtype=np.float32)
    X = X.reshape(-1, 3, 32, 32)
    Xg = X.mean(axis=1)
    mx = float(np.max(Xg))
    if mx > 1.5:
        Xg = Xg / 255.0
    Xg = np.clip(Xg, 0.0, 1.0)
    return (Xg * 255.0 + 0.5).astype(np.uint8)


def stratified_split_idx(y: np.ndarray, test_size: float, seed: int):
    y = np.asarray(y)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=float(test_size), random_state=int(seed))
    tr_idx, va_idx = next(sss.split(np.zeros_like(y), y))
    return tr_idx, va_idx


def dense_sift_descriptors(
    img_u8: np.ndarray,
    *,
    step: int,
    kp_size: float,
    sift: cv2.SIFT,
    max_desc: int | None,
    rng: np.random.Generator,
) -> np.ndarray:
    """Compute dense SIFT descriptors on a regular grid. Returns (n_desc, 128) float32."""
    h, w = img_u8.shape[:2]
    xs = np.arange(step // 2, w, step)
    ys = np.arange(step // 2, h, step)
    keypoints = [cv2.KeyPoint(float(x), float(y), float(kp_size)) for y in ys for x in xs]

    _kps, desc = sift.compute(img_u8, keypoints)
    if desc is None:
        return np.empty((0, 128), dtype=np.float32)

    desc = desc.astype(np.float32, copy=False)
    if max_desc is not None and desc.shape[0] > int(max_desc):
        idx = rng.choice(desc.shape[0], size=int(max_desc), replace=False)
        desc = desc[idx]
    return desc


def sample_descriptors_for_gmm(
    X_u8: np.ndarray,
    *,
    step: int,
    kp_size: float,
    sift: cv2.SIFT,
    max_desc_per_image: int,
    max_total_desc: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    descs = []
    total = 0
    for i in range(X_u8.shape[0]):
        d = dense_sift_descriptors(
            X_u8[i],
            step=step,
            kp_size=kp_size,
            sift=sift,
            max_desc=max_desc_per_image,
            rng=rng,
        )
        if d.shape[0] == 0:
            continue
        descs.append(d)
        total += int(d.shape[0])
        if total >= int(max_total_desc):
            break

    if len(descs) == 0:
        raise RuntimeError("No dense SIFT descriptors found.")

    D = np.vstack(descs)
    if D.shape[0] > int(max_total_desc):
        idx = rng.choice(D.shape[0], size=int(max_total_desc), replace=False)
        D = D[idx]
    return D.astype(np.float32, copy=False)


def fisher_vector(desc: np.ndarray, gmm: GaussianMixture) -> np.ndarray:
    """Improved Fisher Vector (first + second order), diagonal covariances."""
    if desc is None or desc.shape[0] == 0:
        k = int(gmm.n_components)
        d = int(gmm.means_.shape[1])
        return np.zeros((2 * k * d,), dtype=np.float32)

    X = np.asarray(desc, dtype=np.float32)
    w = gmm.weights_.astype(np.float32)
    mu = gmm.means_.astype(np.float32)
    var = gmm.covariances_.astype(np.float32)
    sigma = np.sqrt(var + 1e-12)

    # Responsibilities (N,K)
    Q = gmm.predict_proba(X).astype(np.float32, copy=False)
    N = float(X.shape[0])

    # First order
    X_exp = X[:, None, :]  # (N,1,D)
    mu_exp = mu[None, :, :]  # (1,K,D)
    sigma_exp = sigma[None, :, :]

    diff = (X_exp - mu_exp) / sigma_exp
    u = (Q[:, :, None] * diff).sum(axis=0)  # (K,D)
    u = u / (N * np.sqrt(w)[:, None] + 1e-12)

    # Second order
    diff2 = (diff**2) - 1.0
    v = (Q[:, :, None] * diff2).sum(axis=0)
    v = v / (N * np.sqrt(2.0 * w)[:, None] + 1e-12)

    fv = np.concatenate([u.ravel(), v.ravel()]).astype(np.float32, copy=False)

    # Power normalization + L2
    fv = np.sign(fv) * np.sqrt(np.abs(fv) + 1e-12)
    nrm = float(np.linalg.norm(fv) + 1e-12)
    fv = (fv / nrm).astype(np.float32, copy=False)
    return fv


def build_fisher_features(
    X_u8: np.ndarray,
    *,
    step: int,
    kp_size: float,
    sift: cv2.SIFT,
    gmm: GaussianMixture,
    max_desc_per_image: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    feats = []
    for i in range(X_u8.shape[0]):
        desc = dense_sift_descriptors(
            X_u8[i],
            step=step,
            kp_size=kp_size,
            sift=sift,
            max_desc=max_desc_per_image,
            rng=rng,
        )
        feats.append(fisher_vector(desc, gmm))
    return np.vstack(feats).astype(np.float32, copy=False)


# -----------------------------
# 1) Load data
# -----------------------------
DATA_DIR = "data/"
X_raw, X_test_raw, y_raw = load_data(DATA_DIR)
y_int, inv_map = encode_labels(y_raw)

X_u8 = to_gray_u8_images(X_raw)
X_test_u8 = to_gray_u8_images(X_test_raw)

# -----------------------------
# 2) Reduced tuning
# -----------------------------
seed = 42
val_size = 0.2
subset_size_tune = 1200

X_tune = X_u8[:subset_size_tune]
y_tune = y_int[:subset_size_tune]
tr_idx, va_idx = stratified_split_idx(y_tune, test_size=val_size, seed=seed)

X_tr = X_tune[tr_idx]
X_va = X_tune[va_idx]
y_tr = y_tune[tr_idx]
y_va = y_tune[va_idx]

# Reduced grids
Ks = [16, 32]
steps = [4, 6]
Cs = [0.3, 1.0, 3.0, 10.0]

# SIFT params
kp_size = 8.0
max_desc_per_image_gmm = 96
max_total_desc_gmm = 50000
max_desc_per_image_fv = 128

# Create SIFT extractor once
sift = cv2.SIFT_create()

best = {"acc": -1.0}

print("Running reduced grid search (K, step, C)...")
for K in Ks:
    for step in steps:
        print("=" * 70)
        print({"K": int(K), "step": int(step)})

        print("  Sampling descriptors for GMM...")
        D = sample_descriptors_for_gmm(
            X_tr,
            step=int(step),
            kp_size=kp_size,
            sift=sift,
            max_desc_per_image=max_desc_per_image_gmm,
            max_total_desc=max_total_desc_gmm,
            seed=seed,
        )
        print({"n_desc": int(D.shape[0]), "desc_dim": int(D.shape[1])})

        print("  Fitting GMM...")
        gmm = GaussianMixture(
            n_components=int(K),
            covariance_type="diag",
            max_iter=80,
            reg_covar=1e-6,
            random_state=int(seed),
        )
        gmm.fit(D)

        print("  Building Fisher vectors (train/val)...")
        F_tr = build_fisher_features(
            X_tr,
            step=int(step),
            kp_size=kp_size,
            sift=sift,
            gmm=gmm,
            max_desc_per_image=max_desc_per_image_fv,
            seed=seed,
        )
        F_va = build_fisher_features(
            X_va,
            step=int(step),
            kp_size=kp_size,
            sift=sift,
            gmm=gmm,
            max_desc_per_image=max_desc_per_image_fv,
            seed=seed,
        )

        for C in Cs:
            clf = LinearSVC(C=float(C), dual=True, max_iter=5000)
            clf.fit(F_tr, y_tr)
            acc = float(np.mean(clf.predict(F_va) == y_va))
            print(f"    C={C:<5g}  val_acc={acc:.4f}")
            if acc > best["acc"]:
                best = {
                    "acc": acc,
                    "K": int(K),
                    "step": int(step),
                    "C": float(C),
                }

print("Best params on tuning subset:", best)

# -----------------------------
# 3) Longer final training
# -----------------------------
subset_size_final = int(min(5000, X_u8.shape[0]))
X_sub = X_u8[:subset_size_final]
y_sub = y_int[:subset_size_final]

K = int(best["K"])
step = int(best["step"])
C = float(best["C"])

print("\nFinal training...")
print({"subset_size_final": subset_size_final, "K": K, "step": step, "C": C})

print("  Sampling descriptors for final GMM...")
D_final = sample_descriptors_for_gmm(
    X_sub,
    step=step,
    kp_size=kp_size,
    sift=sift,
    max_desc_per_image=max_desc_per_image_gmm,
    max_total_desc=max_total_desc_gmm,
    seed=seed,
)

print("  Fitting final GMM...")
gmm_final = GaussianMixture(
    n_components=K,
    covariance_type="diag",
    max_iter=120,
    reg_covar=1e-6,
    random_state=int(seed),
)
gmm_final.fit(D_final)

print("  Building Fisher vectors (subset + test)...")
F_sub = build_fisher_features(
    X_sub,
    step=step,
    kp_size=kp_size,
    sift=sift,
    gmm=gmm_final,
    max_desc_per_image=max_desc_per_image_fv,
    seed=seed,
)
F_test = build_fisher_features(
    X_test_u8,
    step=step,
    kp_size=kp_size,
    sift=sift,
    gmm=gmm_final,
    max_desc_per_image=max_desc_per_image_fv,
    seed=seed,
)

print("  Training Linear SVM...")
clf_final = LinearSVC(C=C, dual=True, max_iter=10000)
clf_final.fit(F_sub, y_sub)

yte_int = clf_final.predict(F_test)
yte = np.array([inv_map[int(i)] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
out_path = "results/submission_dense_sift_fisher_linear_svm.csv"
sub.to_csv(out_path, index_label="Id")
print(f"Saved to {out_path}")
print({"subset_size_tune": subset_size_tune, "subset_size_final": subset_size_final, **best})